encoder

In [ ]:
from torch import nn # 从torch库中导入神经网络模块nn

class Encoder(nn.Module): # 定义一个名为Encoder的类，它继承自nn.Module，nn.Module是所有神经网络模块的基类
  def __init__(self, **kwargs): # 构造函数，**kwargs允许传入任意数量的关键字参数
    super(Encoder,self).__init__(**kwargs) # 调用父类nn.Module的构造函数，并传递接收到的所有关键字参数

  def forward(self,x): # 定义前向传播方法，它接收输入x
    raise NotImplementedError # 抛出NotImplementedError，表示这个方法必须由子类实现，当前基类不提供具体实现

decoder

In [ ]:
from torch import nn # 从torch库中导入神经网络模块nn

class Decoder(nn.Module): # 定义一个名为Decoder的类，它继承自nn.Module，nn.Module是所有神经网络模块的基类
  def __init__(self, **kwargs): # 构造函数，**kwargs允许传入任意数量的关键字参数
    super(Decoder, self).__init__(**kwargs) # 调用父类nn.Module的构造函数，并传递接收到的所有关键字参数

  def init_state(self,enc_outputs,*args): # 定义初始化解码器状态的方法，接收编码器输出enc_outputs和可选参数args
    raise NotImplementedError # 抛出NotImplementedError，表示这个方法必须由子类实现，当前基类不提供具体实现

  def forward(self, x): # 定义前向传播方法，它接收输入x
    raise NotImplementedError # 抛出NotImplementedError，表示这个方法必须由子类实现，当前基类不提供具体实现

encoder-decoder

In [ ]:
class EncoderDecoder(nn.Module): # 定义一个名为EncoderDecoder的类，它继承自nn.Module，nn.Module是所有神经网络模块的基类
  def __init__(self, encoder, decoder,**kwargs):
    super(EncoderDecoder, self).__init__(**kwargs) # 调用父类nn.Module的构造函数，并传递接收到的所有关键字参数
    self.encoder = encoder # 将传入的encoder（编码器实例）赋值给self.encoder属性
    self.decoder = decoder # 将传入的decoder（解码器实例）赋值给self.decoder属性

  def forward(self,enc_x,dec_x,*args): # 定义前向传播方法，接收编码器输入enc_x、解码器输入dec_x和可选参数args
    enc_outputs = self.encoder(enc_x,*args) # 调用编码器的forward方法，处理enc_x，得到编码器输出enc_outputs
    dec_state = self.decoder.init_state(enc_outputs,*args) # 调用解码器的init_state方法，使用编码器输出enc_outputs初始化解码器的状态
    return self.decoder(dec_x,dec_state) # 调用解码器的forward方法，处理dec_x和初始化的dec_state，得到最终的解码器输出

seq2seq：从一个句子生成另一个句子，encoder和decoder都是rnn，将encoder
最后时间隐藏状态来初始decoder隐藏状态，来完成信息传递。常用bleu来衡量生成序列的好坏

In [ ]:
!pip install d2l==1.0.0-alpha1.post0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.0/93.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 910.8/910.8 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 104.7 MB/s eta 0:00:00
  Attempting uninstall: jupyter-server
    Found existing installation: jupyter_server 2.18.2
    Uninstalling jupyter_server-2.18.2:
      Successfully uninstalled jupyter_server-2.18.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires jupyter-server==2.18.2, but you have jupyter-server 2.20.

In [ ]:
import collections # 导入collections模块，提供了一些高级的容器数据类型
import math # 导入math模块，提供数学函数
import torch # 导入PyTorch库
from torch import nn # 从torch库中导入神经网络模块nn
from d2l import torch as d2l # 从d2l库中导入PyTorch模块并别名为d2l

# 定义seq2seqEncoder类，继承自d2l库中的Encoder（假设d2l.Encoder是基类）
class seq2seqEncoder(d2l.Encoder):
  # 构造函数，接收词汇表大小、嵌入维度、隐藏单元数量、层数和dropout比率等参数
  def __init__(self,vocab_size,embed_size,num_hiddens,num_layers,dropout=0,**kwargs):
    # 调用父类Encoder的构造函数
    super(seq2seqEncoder,self).__init__(**kwargs)
    # 定义嵌入层，将输入的词汇ID映射为连续的向量表示
    self.embedding = nn.Embedding(vocab_size,embed_size)
    # 定义GRU（Gated Recurrent Unit）层，作为序列到序列编码器的核心循环神经网络
    self.rnn = nn.GRU(embed_size,num_hiddens,num_layers,dropout = dropout)

  # 定义前向传播方法，接收输入x和可选参数args
  def forward(self,x,*args):
    # 将输入x（词汇ID序列）通过嵌入层进行嵌入
    x = self.embedding(x)
    # 调整x的维度顺序，将序列长度放在第一个维度（GRU通常期望的输入格式是(seq_len, batch_size, input_size)）
    x = x.permute(1,0,2)
    # 将处理后的x输入到GRU层，得到输出output和最终隐藏状态state
    output,state = self.rnn(x)
    # 返回GRU的输出和最终隐藏状态
    return output,state

### `seq2seqEncoder` 类参数解释

让我们详细了解 `seq2seqEncoder` 的 `__init__` 方法中的每个参数的含义：

*   **`vocab_size`**：这是词汇表中唯一词元（单词、子词、字符等）的总数。如果您的词汇表中有 10,000 个不同的词，那么 `vocab_size` 将是 10,000。它决定了嵌入矩阵的大小。

*   **`embed_size`**：这是词嵌入的维度。词汇表中的每个词元都将被映射到这个大小的向量。常见的 `embed_size` 可能是 100、256 或 512。更高的维度可以捕获更多的语义信息，但也会增加模型复杂度。

*   **`num_hiddens`**：这指的是 GRU（门控循环单元）层中的隐藏单元（或特征）的数量。它决定了 GRU 每个时间步的隐藏状态向量的大小。这直接影响了模型学习复杂模式的能力。

*   **`num_layers`**：这指定了 GRU 中循环层的数量。多层 GRU（堆叠 GRU）可以学习分层表示，其中较低层捕获基本特征，较高层捕获更抽象的特征。

*   **`dropout`**：这是一种正则化技术。它是一个介于 0 和 1 之间的概率，定义了在训练期间随机设置为零的神经元比例。它通过使模型不那么依赖特定神经元来帮助防止过拟合。

*   **`**kwargs`**：这允许 `Encoder` 基类接收它可能需要的任何额外关键字参数。

####我们也可以通过“翻译官和复习课本”的比喻，来理解五个核心参数：

1. vocab_size（源语言词表大小）
物理意义：代表输入语言一共有多少个不重复的单词

通俗理解：如果我们要做“英文翻译成中文”的任务，而英文数据集里一共出现了 5000 个不重复的英文单词，那么 vocab_size 就是 5000

2. embed_size（词嵌入维度）
物理意义：代表每个单词被压缩成的稠密实数特征向量的长度

通俗理解：我们将一个单词映射为多长的一维数组？通常设为 128、256 或 512。在本例交互看板中，我们将其简化设为 4（每个字由 4 个维度表示）

3. num_hiddens（GRU 隐藏层维度 / 记忆库大小）
物理意义：代表 GRU 循环神经网络内部每个时刻的隐藏状态向量（Hidden State）的长度

通俗理解：这是编码器用于容纳句子历史信息的**“记忆库容量”** 。隐藏维度越大（如 256、512），代表每个时间步能够记录的上下文特征就越丰富。

4. num_layers（GRU 循环层数）
物理意义：代表 GRU 在纵向上重叠堆叠的层数

通俗理解：

如果 num_layers = 1：代表只有一层 GRU 进行特征提取。

如果 num_layers = 2：代表有两层 GRU [2, 3]。第一层 GRU 提取出来的局部序列特征会作为输入，喂入第二层更高级的 GRU 中进行高维特征提炼。

5. dropout（随机丢弃率）
物理意义：在训练期间，以多大的概率随机把 GRU 输出层的一些神经元权重强行置零（不工作），以此防止过拟合

### `seq2seqEncoder` 输入和输出解释

#### `__init__` 方法（初始化）

*   **输入**：上面列出的参数（`vocab_size`、`embed_size`、`num_hiddens`、`num_layers`、`dropout`）。
*   **输出**：一个已初始化的 `seq2seqEncoder` 对象。在内部，它会根据指定的维度设置一个 `nn.Embedding` 层和一个 `nn.GRU` 层。`__init__` 没有返回计算结果意义上的直接“输出”值，而是创建了模型的架构。

#### `forward` 方法（数据流）

*   **输入**：`x`（词元索引张量）和可选的 `*args`。
    *   `x`：这通常是一个形状为 `(batch_size, sequence_length)` 的张量，其中每个元素都是一个整数，表示词汇表中的词元 ID。

*   **中间步骤 1：嵌入层**
    *   `self.embedding(x)` 操作接收输入 `x`（词元 ID）并将每个词元 ID 转换为其对应的密集向量表示。形状从 `(batch_size, sequence_length)` 变为 `(batch_size, sequence_length, embed_size)`。

*   **中间步骤 2：为 GRU 调整维度**
    *   `x.permute(1, 0, 2)` 重塑张量。PyTorch 的 `nn.GRU`（默认情况下）期望输入格式为 `(sequence_length, batch_size, input_size)`。因此，`(batch_size, sequence_length, embed_size)` 变为 `(sequence_length, batch_size, embed_size)`。

*   **中间步骤 3：GRU 层**
    *   `self.rnn(x)` 处理序列。GRU 逐个读取嵌入的词元并更新其内部隐藏状态。

*   **输出**：`output` 和 `state`。
    *   **`output`**：此张量包含 GRU *最后一层* 输入序列 *每个时间步* 的隐藏状态。其形状通常为 `(sequence_length, batch_size, num_hiddens)`。此输出对于需要每个时间步上下文的任务（例如注意力机制）非常有用。
    *   **`state`**：此张量表示 GRU 的 *最终隐藏状态*（如果 `num_layers > 1`，则为多个状态）。对于简单的 GRU，其形状为 `(num_layers, batch_size, num_hiddens)`。这个最终状态通常用作序列到序列模型中解码器的初始隐藏状态，因为它总结了整个输入序列。

### 示例：`seq2seqEncoder` 输入和输出形状

In [ ]:
# 为演示定义虚拟参数
vocab_size = 1000  # 示例：词汇表中有 1000 个唯一的单词
embed_size = 128   # 示例：每个单词由一个 128 维向量表示
num_hiddens = 256  # 示例：GRU 隐藏状态大小为 256
num_layers = 2     # 示例：2 层堆叠 GRU
dropout = 0.1      # 示例：10% 的 dropout

# 实例化 seq2seqEncoder
encoder = seq2seqEncoder(
    vocab_size=vocab_size,
    embed_size=embed_size,
    num_hiddens=num_hiddens,
    num_layers=num_layers,
    dropout=dropout
)

print("Encoder 实例化成功！")

# 准备一个虚拟输入张量
batch_size = 32         # 示例：一个批次有 32 个句子
sequence_length = 10    # 示例：每个句子有 10 个词元

# 创建一个随机词元 ID 张量（0 到 vocab_size-1 之间的整数）
# 形状: (batch_size, sequence_length)
x_input = torch.randint(0, vocab_size, (batch_size, sequence_length))

print(f"\n--- `forward` 方法的输入 `x` ---")
print(f"形状: {x_input.shape}") # 预期: (32, 10)
print(f"数据类型: {x_input.dtype}") # 预期: torch.int64

# 执行前向传播
output, state = encoder(x_input)

print(f"\n--- `forward` 方法的输出 ---")
print(f"输出 (最后一层的所有隐藏状态):")
print(f"  形状: {output.shape}") # 预期: (sequence_length, batch_size, num_hiddens) 即 (10, 32, 256)
print(f"  数据类型: {output.dtype}") # 预期: torch.float32

print(f"状态 (所有层的最终隐藏状态):")
print(f"  形状: {state.shape}") # 预期: (num_layers, batch_size, num_hiddens) 即 (2, 32, 256)
print(f"  数据类型: {state.dtype}") # 预期: torch.float32


Encoder 实例化成功！

--- `forward` 方法的输入 `x` ---
形状: torch.Size([32, 10])
数据类型: torch.int64

--- `forward` 方法的输出 ---
输出 (最后一层的所有隐藏状态):
  形状: torch.Size([10, 32, 256])
  数据类型: torch.float32
状态 (所有层的最终隐藏状态):
  形状: torch.Size([2, 32, 256])
  数据类型: torch.float32


可视化

In [ ]:
# @title
from IPython.display import HTML

html_code = r"""
<!DOCTYPE html>
<html>
<head>
  <!-- 引入 Tailwind CSS 进行界面美化 -->
  <script src="https://cdn.tailwindcss.com"></script>
  <style>
    @import url('https://fonts.googleapis.com/css2?family=Noto+Sans+SC:wght@400;500;700&display=swap');
    body { font-family: 'Noto Sans SC', sans-serif; }
  </style>
</head>
<body class="bg-slate-950 text-slate-100 flex items-center justify-center p-4">
  <div class="w-full max-w-2xl bg-slate-900 border border-slate-800 rounded-3xl p-6 shadow-2xl">
    <h2 class="text-lg font-bold text-center text-cyan-400 mb-2">Seq2Seq Encoder 数据流转与维度幻化</h2>
    <p class="text-xs text-slate-400 text-center mb-6">点击下方计算步骤，观察输入 X 是如何在各层之间改变形状（Shape）的</p>

    <!-- 1. 步骤控制按钮 -->
    <div class="flex gap-2 p-1 bg-slate-950 rounded-xl mb-6 text-xs">
      <button id="btn-s1" onclick="showStep(1)" class="flex-1 py-2 px-1.5 font-semibold rounded-lg bg-cyan-600 text-white focus:outline-none transition-all duration-300">
        1. 原始输入 x
      </button>
      <button id="btn-s2" onclick="showStep(2)" class="flex-1 py-2 px-1.5 font-semibold rounded-lg text-slate-400 hover:text-white focus:outline-none transition-all duration-300">
        2. Embedding 词嵌入
      </button>
      <button id="btn-s3" onclick="showStep(3)" class="flex-1 py-2 px-1.5 font-semibold rounded-lg text-slate-400 hover:text-white focus:outline-none transition-all duration-300">
        3. Permute 换轴
      </button>
      <button id="btn-s4" onclick="showStep(4)" class="flex-1 py-2 px-1.5 font-semibold rounded-lg text-slate-400 hover:text-white focus:outline-none transition-all duration-300">
        4. GRU 最终计算
      </button>
    </div>

    <!-- 2. 张量形状展示网格 -->
    <div class="grid grid-cols-1 md:grid-cols-2 gap-4 mb-6">
      <!-- 当前执行的代码 -->
      <div class="p-4 bg-slate-950/60 border border-slate-800 rounded-2xl">
        <span class="text-[10px] font-bold text-slate-500 uppercase block mb-1">当前执行的代码（Python）</span>
        <code id="code-display" class="text-xs font-mono text-cyan-300 block leading-relaxed h-12 flex items-center">
          <!-- 动态注入 -->
        </code>
      </div>
      <!-- 产生的形状 Shape -->
      <div class="p-4 bg-slate-950/60 border border-slate-800 rounded-2xl">
        <span class="text-[10px] font-bold text-slate-500 uppercase block mb-1">产生的张量形状 (Shape)</span>
        <div id="shape-display" class="text-sm font-bold font-mono text-emerald-400 h-12 flex items-center">
          <!-- 动态注入 -->
        </div>
      </div>
    </div>

    <!-- 3. 直观大白话原理解析 -->
    <div class="p-4 bg-slate-950/50 rounded-2xl border border-slate-800 text-xs">
      <span id="explain-title" class="text-xs font-bold text-amber-400 block mb-1.5">1. 原始输入 x</span>
      <p id="explain-text" class="text-slate-300 leading-relaxed">
        我们一次传入了 2 个句子（Batch_size=2），每句话有 3 个单词（Seq_len=3）...
      </p>
    </div>
  </div>

  <script>
    // 定义步骤的配置文件，避开了 Python string 冲突
    const config = {
      1: {
        code: "x = torch.tensor([[3, 5, 2], [1, 9, 4]])",
        shape: "torch.Size([2, 3]) <br><span class='text-[10px] text-slate-500'>[Batch_size, Seq_len]</span>",
        title: "1. 原始输入 x (词汇 ID 矩阵)",
        text: "我们一次从发牌机里抽出了 <b>2 个句子</b>（Batch_size=2），每句话包含 <b>3 个单词</b>（Seq_len=3） [2]。<br>矩阵里的数字（如 3, 5, 2）就是这些单词在字典（Vocabulary）里的 <b>整数 ID 编号</b> [2, 3]。"
      },
      2: {
        code: "x = self.embedding(x)",
        shape: "torch.Size([2, 3, 4]) <br><span class='text-[10px] text-slate-500'>[Batch, Seq_len, Embed_size]</span>",
        title: "2. 词嵌入 (Embedding Layer)",
        text: "数据通过嵌入层，每个原本干瘪的整数 ID，都被自动翻译成了一个 <b>4 维的稠密实数特征向量</b>（embed_size=4） [2, 3]。<br>现在数据升维成了 3D 张量，完美携带了每个词的语义特征 [2, 3]。"
      },
      3: {
        code: "x = x.permute(1, 0, 2)",
        shape: "torch.Size([3, 2, 4]) <br><span class='text-[10px] text-slate-500'>[Seq_len, Batch, Embed_size]</span>",
        title: "3. 维度重排 (Permute)",
        text: "这是 PyTorch 循环神经网络的一个**极其重要的特征**！<br>将第 0 维（Batch）和第 1 维（Seq_len）进行了<b>互换</b>。<br>因为 PyTorch 的 GRU 默认要求输入的第一维必须是<b>时间步（Seq_len）</b>，以便它能按照时间先后，并行且顺序地处理数据 [2]。"
      },
      4: {
        code: "output, state = self.rnn(x)",
        shape: "• output: torch.Size([3, 2, 8]) <br>• state : torch.Size([1, 2, 8])",
        title: "4. GRU 循环层最终计算 (RNN Forward)",
        text: "数据喂入隐藏维度为 8（num_hiddens=8）的 1 层 GRU 中计算，吐出两个高能变量 [2, 3]：<br>• <b>output [3, 2, 8]</b>：包含<b>每个时间步</b>对应的隐藏状态（给注意力机制用） [2, 3]。<br>• <b>state [1, 2, 8]</b>：只包含<b>最末尾那个时间步</b>的最终隐藏状态（即整句记忆向量），直接丢给 Decoder 用 [2, 3]。"
      }
    };

    function showStep(s) {
      // 1. 更新按钮样式
      for (let i = 1; i <= 4; i++) {
        const btn = document.getElementById(`btn-s${i}`);
        if (i === s) {
          btn.className = "flex-1 py-2 px-1.5 font-semibold rounded-lg bg-cyan-600 text-white focus:outline-none transition-all duration-300";
        } else {
          btn.className = "flex-1 py-2 px-1.5 font-semibold rounded-lg text-slate-400 hover:text-white focus:outline-none transition-all duration-300";
        }
      }

      // 2. 动态修改面板内容
      const info = config[s];
      document.getElementById('code-display').innerHTML = info.code;
      document.getElementById('shape-display').innerHTML = info.shape;
      document.getElementById('explain-title').innerHTML = info.title;
      document.getElementById('explain-text').innerHTML = info.text;
    }

    // 默认点亮第一步
    showStep(1);
  </script>
</body>
</html>
"""

# 一键渲染到 Colab 单元格
from IPython.display import HTML
HTML(html_code)

尝试实现一个seq2seq-encoder

In [ ]:
import collections # 导入collections模块，提供了一些高级的容器数据类型
import math # 导入math模块，提供数学函数
import torch # 导入PyTorch库
from torch import nn # 从torch库中导入神经网络模块nn
from d2l import torch as d2l # 从d2l库中导入PyTorch模块并别名为d2l

# 定义seq2seqEncoder类，继承自d2l库中的Encoder（假设d2l.Encoder是基类）
class seq2seqEncoder(d2l.Encoder):
  # 构造函数，接收词汇表大小、嵌入维度、隐藏单元数量、层数和dropout比率等参数
  def __init__(self,vocab_size,embed_size,num_hiddens,num_layers,dropout=0,**kwargs):
    # 调用父类Encoder的构造函数
    super(seq2seqEncoder,self).__init__(**kwargs)
    # 定义嵌入层，将输入的词汇ID映射为连续的向量表示
    self.embedding = nn.Embedding(vocab_size,embed_size)
    # 定义GRU（Gated Recurrent Unit）层，作为序列到序列编码器的核心循环神经网络
    self.rnn = nn.GRU(embed_size,num_hiddens,num_layers,dropout = dropout)

  # 定义前向传播方法，接收输入x和可选参数args
  def forward(self,x,*args):
    # 将输入x（词汇ID序列）通过嵌入层进行嵌入
    x = self.embedding(x)
    # 调整x的维度顺序，将序列长度放在第一个维度（GRU通常期望的输入格式是(seq_len, batch_size, input_size)）
    x = x.permute(1,0,2)
    # 将处理后的x输入到GRU层，得到输出output和最终隐藏状态state
    output,state = self.rnn(x)
    # 返回GRU的输出和最终隐藏状态
    return output,state

In [ ]:
# 实例化编码器：词表大小10，嵌入维度6，隐藏单元1，GRU层数2
encoder = seq2seqEncoder(vocab_size=10, embed_size=6, num_hiddens=1, num_layers=2)

# 将模型设置为评估模式（关闭 dropout 等）
encoder.eval()

# 创建虚拟输入 x：形状为 (batch_size=4, seq_len=7) 的全零长整型张量
x = torch.zeros((4, 7), dtype=torch.long)

# 执行前向传播，得到输出和最终状态
output, state = encoder(x)

# 查看输出的形状：预期为 (seq_len=7, batch_size=4, num_hiddens=1)
print(output.shape)

# 查看最终状态的形状：预期为 (num_layers=2, batch_size=4, num_hiddens=1)
print(state.shape)

torch.Size([7, 4, 1])
torch.Size([2, 4, 1])


可视化

In [ ]:
# @title
from IPython.display import HTML

html_code = r"""
<!DOCTYPE html>
<html>
<head>
  <script src="https://cdn.tailwindcss.com"></script>
  <style>
    @import url('https://fonts.googleapis.com/css2?family=Noto+Sans+SC:wght@400;500;700&display=swap');
    body {
      font-family: 'Noto Sans SC', sans-serif;
      background-color: #0a0c12;
      color: #dde1f0;
    }
    .tab-content { transition: all 0.3s ease; }
    .grid-cell { transition: all 0.3s ease; }
  </style>
</head>
<body class="p-4 md:p-6 min-h-screen flex items-center justify-center">

  <div class="w-full max-w-2xl bg-[#12151e] border border-[#1e2235] rounded-3xl p-6 shadow-2xl">

    <!-- 头部信息 -->
    <div class="text-center mb-6">
      <div class="text-[10px] text-[#8b7cf8] tracking-[4px] uppercase mb-1.5 font-bold">d2l · Seq2Seq</div>
      <h1 class="text-2xl font-extrabold text-white">Encoder 代码详解</h1>
      <div class="text-xs text-[#4a5068] mt-1.5">Embedding + 多层 GRU 计算流转</div>
    </div>

    <!-- 1. 主导航 Tabs (4个) -->
    <div class="flex gap-2 p-1 bg-[#0a0c12] rounded-xl mb-6 flex-wrap text-xs">
      <button id="tab-btn-structure" onclick="switchTab('structure')" class="flex-1 py-2 px-3 font-bold rounded-lg bg-[#8b7cf8] text-white focus:outline-none transition-all">
        🏗️ 整体结构
      </button>
      <button id="tab-btn-dim" onclick="switchTab('dim')" class="flex-1 py-2 px-3 font-bold rounded-lg text-[#4a5068] hover:text-white focus:outline-none transition-all">
        📐 维度变化
      </button>
      <button id="tab-btn-outstate" onclick="switchTab('outstate')" class="flex-1 py-2 px-3 font-bold rounded-lg text-[#4a5068] hover:text-white focus:outline-none transition-all">
        🔑 output vs state
      </button>
      <button id="tab-btn-example" onclick="switchTab('example')" class="flex-1 py-2 px-3 font-bold rounded-lg text-[#4a5068] hover:text-white focus:outline-none transition-all">
        🔢 例子验证
      </button>
    </div>

    <!-- ========================================== -->
    <!-- Tab 1: 整体结构 (Structure) -->
    <!-- ========================================== -->
    <div id="tab-structure" class="tab-content space-y-4">
      <!-- Card 1 -->
      <div class="bg-[#12151e] border border-[#1e2235] rounded-2xl p-5 space-y-3">
        <div class="text-sm font-bold text-[#22d3ee]">整体结构：Embedding + 多层 GRU</div>
        <p class="text-xs text-[#4a5068] leading-relaxed">这个 Encoder 只有两个组件，比之前手写的普通 RNN 多了一个“层数（Layer）”和“自适应门控”的精神理念。</p>

        <div class="space-y-2.5">
          <div class="bg-[#0a0c12] rounded-xl p-3.5 border border-[#22d3ee]/20">
            <div class="text-xs font-bold text-[#22d3ee] mb-1">① nn.Embedding(vocab_size, embed_size)</div>
            <p class="text-[11px] text-[#4a5068]">词 id ──> 向量。这里词表有 10 个词，每个词进入后自动转化为 6 维的特征向量。</p>
          </div>
          <div class="bg-[#0a0c12] rounded-xl p-3.5 border border-[#8b7cf8]/20">
            <div class="text-xs font-bold text-[#8b7cf8] mb-1">② nn.GRU(embed_size, num_hiddens, num_layers)</div>
            <p class="text-[11px] text-[#4a5068]">比普通 RNN 多了“门控（Gate）机制”，能自适应决定记住/忘记什么；num_layers=2 表示纵向重叠了 2 层 GRU。</p>
          </div>
        </div>
      </div>

      <!-- Card 2 -->
      <div class="bg-[#12151e] border border-[#1e2235] rounded-2xl p-5 space-y-3">
        <div class="text-sm font-bold text-[#dde1f0]">GRU vs. 普通 RNN（你之前学过的）</div>
        <div class="grid grid-cols-1 md:grid-cols-2 gap-3">
          <div class="bg-[#0a0c12] rounded-xl p-3 border border-[#4a5068]/30">
            <span class="text-[10px] text-[#4a5068] block mb-1">普通 RNN（手写版）</span>
            <code class="text-xs text-[#dde1f0] font-mono">h(t) = tanh(W_x·x + W_h·h_prev)</code>
            <p class="text-[9px] text-[#f87171] mt-2">⚠️ 致命痛点：长序列容易发生“梯度消失/早期遗忘”</p>
          </div>
          <div class="bg-[#0a0c12] rounded-xl p-3 border border-[#8b7cf8]/30">
            <span class="text-[10px] text-[#8b7cf8] block mb-1">GRU（门控循环单元）</span>
            <code class="text-xs text-[#dde1f0] font-mono leading-relaxed">z=更新门, r=重置门<br>h(t) = (1-z)·h_prev + z·h_候选</code>
            <p class="text-[9px] text-[#4ade80] mt-2">✅ 物理优势：能自动学会“该记住什么，该忘记什么”</p>
          </div>
        </div>
      </div>

      <!-- Card 3 -->
      <div class="bg-[#12151e] border border-[#1e2235] rounded-2xl p-5 space-y-4">
        <div class="text-sm font-bold text-[#fb923c]">什么是“多层” GRU (num_layers=2)？</div>
        <div class="flex items-center justify-center gap-4 text-xs font-bold py-2 bg-[#0a0c12] rounded-xl">
          <div class="text-center">
            <span class="text-[9px] text-[#4a5068] block mb-1">输入</span>
            <span class="bg-[#22d3ee]/10 text-[#22d3ee] border border-[#22d3ee]/40 px-3 py-1.5 rounded-lg block font-mono">Embed</span>
          </div>
          <span class="text-[#4a5068] text-base">➔</span>
          <div class="text-center">
            <span class="text-[9px] text-[#8b7cf8] block mb-1">第 1 层 GRU</span>
            <span class="bg-[#8b7cf8]/10 text-[#8b7cf8] border border-[#8b7cf8]/40 px-3 py-1.5 rounded-lg block font-mono">Layer 0</span>
          </div>
          <span class="text-[#4a5068] text-base">➔</span>
          <div class="text-center">
            <span class="text-[9px] text-[#4ade80] block mb-1">第 2 层 GRU</span>
            <span class="bg-[#4ade80]/10 text-[#4ade80] border border-[#4ade80]/40 px-3 py-1.5 rounded-lg block font-mono">Layer 1</span>
          </div>
        </div>
        <p class="text-xs text-[#4a5068] leading-relaxed">
          第 1 层的输出直接作为第 2 层的输入 —— 就像叠了两个不同维度的“漏斗”。
          第 1 层负责捕获低级基础规律，第 2 层在第 1 层的基础上提炼、合并更高级的抽象语义特征。
        </p>
      </div>
    </div>

    <!-- ========================================== -->
    <!-- Tab 2: 维度变化 (Dim Flow) -->
    <!-- ========================================== -->
    <div id="tab-dim" class="tab-content space-y-4 hidden">
      <div class="bg-[#12151e] border border-[#1e2235] rounded-2xl p-5">
        <div class="text-sm font-bold text-[#dde1f0] mb-4">完整维度流转折线图</div>

        <!-- Timeline Steps -->
        <div class="space-y-4">
          <!-- Step 1 -->
          <div class="flex">
            <div class="flex flex-col items-center w-8 mr-3">
              <span class="w-6 h-6 rounded-full bg-[#22d3ee]/10 border-2 border-[#22d3ee] flex items-center justify-center text-xs font-black text-[#22d3ee] z-10">1</span>
              <div class="w-0.5 bg-[#1e2235] flex-1 my-1"></div>
            </div>
            <div class="bg-[#0a0c12] border border-[#22d3ee]/20 rounded-xl p-3 flex-1 flex justify-between items-center">
              <div>
                <span class="text-xs font-bold text-[#22d3ee] block">1. 原始输入 x (词汇 ID 矩阵)</span>
                <span class="text-[10px] text-[#4a5068]">batch=4个句子, seq_len=7个单词</span>
              </div>
              <code class="text-xs font-bold text-[#22d3ee] font-mono">(4, 7)</code>
            </div>
          </div>
          <!-- Step 2 -->
          <div class="flex">
            <div class="flex flex-col items-center w-8 mr-3">
              <span class="w-6 h-6 rounded-full bg-purple-500 border border-purple-500 flex items-center justify-center text-xs font-bold text-white">2</span>
              <div class="border-l border-slate-800 h-full"></div>
            </div>
            <div class="bg-[#0a0c12] border border-purple-500/20 rounded-xl p-3 flex-1 flex justify-between items-center">
              <div>
                <span class="text-xs font-bold text-purple-400 block">2. self.embedding(x) 词嵌入</span>
                <span class="text-[10px] text-[#4a5068]">每个词 id 映射为 6 维稠密特征向量</span>
              </div>
              <code class="text-xs font-bold text-purple-400 font-mono">(4, 7, 6)</code>
            </div>
          </div>
          <!-- Step 3 -->
          <div class="flex flex-col items-center">
            <div class="flex w-full items-center">
              <div class="w-10 flex flex-col items-center">
                <span class="w-6 h-6 rounded-full border border-amber-500 bg-amber-950/50 flex items-center justify-center text-xs font-bold text-amber-300">3</span>
                <div class="w-0.5 bg-[#1e2235] h-8 my-1"></div>
              </div>
              <div class="bg-[#0a0c12] border border-amber-500/20 rounded-xl p-3 flex-1 flex justify-between items-center ml-1">
                <div>
                  <span class="text-xs font-bold text-amber-400 block">3. x.permute(1, 0, 2) 换轴</span>
                  <span class="text-[10px] text-[#4a5068]">将时间步（seq_len）挪到第 0 维度</span>
                </div>
                <code class="text-xs font-bold text-amber-400 font-mono">(7, 4, 6)</code>
              </div>
            </div>
          </div>
          <!-- Step 4 -->
          <div class="flex">
            <div class="flex flex-col items-center w-8 mr-3">
              <span class="w-6 h-6 rounded-full bg-[#4ade80]/10 border-2 border-[#4ade80] flex items-center justify-center text-xs font-black text-[#4ade80] z-10">4</span>
              <div class="w-0.5 bg-[#1e2235] h-6 my-1"></div>
            </div>
            <div class="bg-[#0a0c12] border border-[#4ade80]/20 rounded-xl p-3 flex-1 flex justify-between items-center">
              <div>
                <span class="text-xs font-bold text-[#4ade80] block">4. GRU 输出：output</span>
                <span class="text-[10px] text-[#4a5068]">最顶层 GRU，每个时刻的隐藏状态输出</span>
              </div>
              <code class="text-xs font-bold text-[#4ade80] font-mono">(7, 4, 1)</code>
            </div>
          </div>
          <!-- Step 5 -->
          <div class="flex">
            <div class="flex flex-col items-center w-8 mr-3">
              <span class="w-6 h-6 rounded-full bg-[#fbbf24]/10 border-2 border-[#fbbf24] flex items-center justify-center text-xs font-black text-[#fbbf24] z-10">5</span>
            </div>
            <div class="bg-[#0a0c12] border border-[#fbbf24]/20 rounded-xl p-3 flex-1 flex justify-between items-center">
              <div>
                <span class="text-xs font-bold text-[#fbbf24] block">5. GRU 状态：state</span>
                <span class="text-[10px] text-[#4a5068]">整句读完时，每一层 GRU 各自的最终状态</span>
              </div>
              <code class="text-xs font-bold text-[#fbbf24] font-mono">(2, 4, 1)</code>
            </div>
          </div>
        </div>
      </div>

      <!-- Card 2 -->
      <div class="bg-[#12151e] border border-[#1e2235] rounded-2xl p-5 space-y-3">
        <div class="text-sm font-bold text-[#fb923c]">为什么要进行 permute(1, 0, 2) 换轴？</div>
        <p class="text-xs text-[#4a5068] leading-relaxed">
          PyTorch 的 `nn.GRU` 底层默认要求输入的 3D 数据格式必须为：
          <code class="text-amber-400 font-mono font-bold">(seq_len, batch_size, input_size)</code>。<br>
          然而词嵌入层出来的格式是：
          <code class="text-[#22d3ee] font-mono font-bold">(batch_size, seq_len, input_size)</code>。<br>
          为了对齐框架，我们必须使用 `permute` 将第 0 维和第 1 维对换！
        </p>
        <div class="flex items-center justify-center gap-3 text-xs bg-[#0a0c12] py-2 rounded-xl">
          <span class="bg-[#22d3ee]/10 text-[#22d3ee] px-2 py-0.5 rounded border border-[#22d3ee]/20 font-mono">(batch, seq, feat)</span>
          <span class="text-[#4a5068]">➔</span>
          <span class="bg-[#fb923c]/10 text-[#fb923c] px-2 py-0.5 rounded border border-[#fb923c]/20 font-mono">(seq, batch, feat)</span>
        </div>
      </div>
    </div>

    <!-- ========================================== -->
    <!-- Tab 3: output vs state (Core Concept) -->
    <!-- ========================================== -->
    <div id="tab-outstate" class="tab-content space-y-4 hidden">
      <div class="bg-[#12151e] border border-[#1e2235] rounded-2xl p-5">
        <div class="text-sm font-bold text-[#fbbf24] mb-2">output 和 state 到底是什么？（极重要）</div>
        <p class="text-xs text-[#4a5068] leading-relaxed mb-4">
          把“层数（Layer）”和“时间步（Time Step）”想象成一个<b>二维网格</b>。
          网格里的每个格子都是一个隐藏状态 $h$。`output` 和 `state` 实际上是这个网格中<b>两个完全不同的切片</b>方式：
        </p>

        <!-- Toggle Buttons -->
        <div class="flex gap-2 p-1 bg-[#0a0c12] rounded-xl mb-6 text-xs">
          <button id="btn-hl-output" onclick="setHighlight('output')" class="flex-1 py-2 px-3 font-bold rounded-lg bg-[#4ade80]/20 text-[#4ade80] border border-[#4ade80] focus:outline-none transition-all">
            查看 output 切片
          </button>
          <button id="btn-hl-state" onclick="setHighlight('state')" class="flex-1 py-2 px-3 font-bold rounded-lg text-[#4a5068] hover:text-white focus:outline-none transition-all">
            查看 state 切片
          </button>
        </div>

        <!-- Matrix Grid -->
        <div class="overflow-x-auto py-2">
          <div class="inline-flex flex-col gap-2 font-mono text-[10px]">
            <!-- Layer 1 (Top) -->
            <div class="flex items-center gap-1.5">
              <span class="w-20 text-right text-[#4a5068]">Layer 1 (顶层)</span>
              <div id="g-1-1" class="grid-cell w-9 h-9 border border-[#1e2235] bg-[#0a0c12] rounded-lg flex items-center justify-center text-slate-500">h</div>
              <div id="g-1-2" class="grid-cell w-9 h-9 border border-[#1e2235] bg-[#0a0c12] rounded-lg flex items-center justify-center text-slate-500">h</div>
              <div id="g-1-3" class="grid-cell w-9 h-9 border border-[#1e2235] bg-[#0a0c12] rounded-lg flex items-center justify-center text-slate-500">h</div>
              <div id="g-1-4" class="grid-cell w-9 h-9 border border-[#1e2235] bg-[#0a0c12] rounded-lg flex items-center justify-center text-slate-500">h</div>
              <div id="g-1-5" class="grid-cell w-9 h-9 border border-[#1e2235] bg-[#0a0c12] rounded-lg flex items-center justify-center text-slate-500">h</div>
              <div id="g-1-6" class="grid-cell w-9 h-9 border border-[#1e2235] bg-[#0a0c12] rounded-lg flex items-center justify-center text-slate-500">h</div>
              <div id="g-1-7" class="grid-cell w-9 h-9 border border-[#1e2235] bg-[#0a0c12] rounded-lg flex items-center justify-center text-slate-500">h</div>
            </div>
            <!-- Layer 0 (Bottom) -->
            <div class="flex items-center gap-1.5">
              <span class="w-20 text-right text-[#4a5068]">Layer 0 (底层)</span>
              <div id="g-0-1" class="grid-cell w-9 h-9 border border-[#1e2235] bg-[#0a0c12] rounded-lg flex items-center justify-center text-slate-500">h</div>
              <div id="g-0-2" class="grid-cell w-9 h-9 border border-[#1e2235] bg-[#0a0c12] rounded-lg flex items-center justify-center text-slate-500">h</div>
              <div id="g-0-3" class="grid-cell w-9 h-9 border border-[#1e2235] bg-[#0a0c12] rounded-lg flex items-center justify-center text-slate-500">h</div>
              <div id="g-0-4" class="grid-cell w-9 h-9 border border-[#1e2235] bg-[#0a0c12] rounded-lg flex items-center justify-center text-slate-500">h</div>
              <div id="g-0-5" class="grid-cell w-9 h-9 border border-[#1e2235] bg-[#0a0c12] rounded-lg flex items-center justify-center text-slate-500">h</div>
              <div id="g-0-6" class="grid-cell w-9 h-9 border border-[#1e2235] bg-[#0a0c12] rounded-lg flex items-center justify-center text-slate-500">h</div>
              <div id="g-0-7" class="grid-cell w-9 h-9 border border-[#1e2235] bg-[#0a0c12] rounded-lg flex items-center justify-center text-slate-500">h</div>
            </div>
            <!-- Time axis -->
            <div class="flex gap-1.5 ml-20">
              <span class="w-9 text-center text-[#4a5068] text-[9px]">t1</span>
              <span class="w-9 text-center text-[#4a5068] text-[9px]">t2</span>
              <span class="w-9 text-center text-[#4a5068] text-[9px]">t3</span>
              <span class="w-9 text-center text-[#4a5068] text-[9px]">t4</span>
              <span class="w-9 text-center text-[#4a5068] text-[9px]">t5</span>
              <span class="w-9 text-center text-[#4a5068] text-[9px]">t6</span>
              <span class="w-9 text-center text-[#4a5068] text-[9px]">t7 (末尾)</span>
            </div>
          </div>
        </div>

        <!-- Explainer Alert -->
        <div id="outstate-alert" class="mt-4 p-3 rounded-xl bg-[#4ade80]/10 border border-[#4ade80]/30 text-xs text-[#4ade80] leading-relaxed">
          <!-- 动态注入 -->
        </div>
      </div>

      <!-- Card 2 -->
      <div class="bg-[#12151e] border border-[#1e2235] rounded-2xl p-5 space-y-2">
        <div class="text-xs font-bold text-[#dde1f0]">🔍 一个隐藏极深的数学小细节</div>
        <p class="text-[11px] text-[#4a5068] leading-relaxed">
          `output[-1]`（output 最后一个时刻）与 `state[-1]`（state 最后一层）**指向的是同一个隐藏状态特征**！<br>
          它们都对应着“最顶层（Layer 1）、最后一个时间步（t7）”的那个交叉格子，只是由于切片提取的视角不同，被包裹在不同的张量维度中。
        </p>
      </div>

      <!-- Card 3 -->
      <div class="bg-[#12151e] border border-[#1e2235] rounded-2xl p-4 text-[11px] text-[#4a5068] leading-relaxed">
        在后续做翻译（Decoder）时：<br>
        • <b>`state`</b> 会被直接传给 Decoder 当作初始的“记忆包”（因为它保留了每一层的终极印记） [2]；<br>
        • <b>`output`</b> 则被注意力（Attention）机制去扫视（因为 Attention 需要看清输入句子每一步的信息，不能只看最后一步） [3]。
      </div>
    </div>

    <!-- ========================================== -->
    <!-- Tab 4: 例子验证 (Example) -->
    <!-- ========================================== -->
    <div id="tab-example" class="tab-content space-y-4 hidden">
      <div class="bg-[#12151e] border border-[#1e2235] rounded-2xl p-5 space-y-4">
        <div class="text-sm font-bold text-[#dde1f0]">代入实例中的真实参数</div>
        <div class="grid grid-cols-2 gap-3 text-[10px]">
          <div class="bg-[#0a0c12] border border-[#22d3ee]/20 p-2.5 rounded-xl">
            <span class="bg-[#22d3ee]/10 text-[#22d3ee] px-1.5 py-0.5 rounded border border-[#22d3ee]/20 font-bold block mb-1">vocab_size=10</span>
            <span class="text-[#4a5068]">词汇表一共有 10 个数字。</span>
          </div>
          <div class="bg-[#0a0c12] border border-purple-500/20 p-2.5 rounded-xl">
            <span class="bg-purple-500/10 text-purple-400 px-1.5 py-0.5 rounded border border-purple-500/20 font-bold block mb-1">embed_size=6</span>
            <span class="text-[#4a5068]">每个单词特征向量大小为 6。</span>
          </div>
          <div class="bg-[#0a0c12] border border-orange-500/20 p-2.5 rounded-xl">
            <span class="bg-orange-500/10 text-orange-400 px-1.5 py-0.5 rounded border border-orange-500/20 font-bold block mb-1">num_hiddens=1</span>
            <span class="text-[#4a5068]">隐藏状态设为 1，方便数维度。</span>
          </div>
          <div class="bg-[#0a0c12] border border-emerald-500/20 p-2.5 rounded-xl">
            <span class="bg-emerald-500/10 text-emerald-400 px-1.5 py-0.5 rounded border border-emerald-500/20 font-bold block mb-1">num_layers=2</span>
            <span class="text-[#4a5068]">纵向叠 2 层 GRU 进行处理 [2]。</span>
          </div>
          <div class="bg-[#0a0c12] border border-yellow-500/20 p-2.5 rounded-xl">
            <span class="bg-yellow-500/10 text-yellow-400 px-1.5 py-0.5 rounded border border-yellow-500/20 font-bold block mb-1">batch_size=4</span>
            <span class="text-[#4a5068]">一次性并行读 4 个句子 [3]。</span>
          </div>
          <div class="bg-[#0a0c12] border border-pink-500/20 p-2.5 rounded-xl">
            <span class="bg-pink-500/10 text-pink-400 px-1.5 py-0.5 rounded border border-pink-500/20 font-bold block mb-1">seq_len=7</span>
            <span class="text-[#4a5068]">每句话有 7 个单词长。</span>
          </div>
        </div>
      </div>

      <!-- Verification box -->
      <div class="bg-[#12151e] border border-[#1e2235] rounded-2xl p-5 space-y-3">
        <div class="text-sm font-bold text-emerald-400">控制台打印出的最终 Shape 对齐验证</div>
        <pre class="bg-[#0a0c12] text-xs font-mono p-4 rounded-xl leading-relaxed text-[#dde1f0]">
output.shape = (<span class="text-pink-400">7</span>, <span class="text-yellow-400">4</span>, <span class="text-orange-400">1</span>)
                ↑seq  ↑batch ↑hidden

state.shape  = (<span class="text-emerald-400">2</span>, <span class="text-yellow-400">4</span>, <span class="text-orange-400">1</span>)
                ↑layers ↑batch ↑hidden
        </pre>
      </div>
    </div>

  </div>

  <script>
    // 1. 主 Tab 切换逻辑
    const tabContents = ['structure', 'dim', 'outstate', 'example'];
    function switchTab(activeId) {
      tabContents.forEach(id => {
        const btn = document.getElementById(`tab-btn-${id}`);
        const content = document.getElementById(`tab-${id}`);
        if (id === activeId) {
          btn.className = "flex-1 py-2 px-3 font-bold rounded-lg bg-[#8b7cf8] text-white focus:outline-none transition-all";
          content.classList.remove('hidden');
        } else {
          btn.className = "flex-1 py-2 px-3 font-bold rounded-lg text-[#4a5068] hover:text-white focus:outline-none transition-all";
          content.classList.add('hidden');
        }
      });
      // 触发一下，确保第三个tab打开时样式正确初始化
      if (activeId === 'outstate') {
        setHighlight('output');
      }
    }

    // 2. output vs state 网格高亮控制
    function setHighlight(type) {
      const btnOut = document.getElementById('btn-hl-output');
      const btnState = document.getElementById('btn-hl-state');
      const alertBox = document.getElementById('outstate-alert');

      // 清除所有格子样式
      for (let l = 0; l < 2; l++) {
        for (let t = 1; t <= 7; t++) {
          const cell = document.getElementById(`g-${l}-${t}`);
          cell.className = "grid-cell w-9 h-9 border border-[#1e2235] bg-[#0a0c12] rounded-lg flex items-center justify-center text-slate-500";
        }
      }

      if (type === 'output') {
        btnOut.className = "flex-1 py-2 px-3 font-bold rounded-lg bg-[#4ade80]/20 text-[#4ade80] border border-[#4ade80] focus:outline-none transition-all";
        btnState.className = "flex-1 py-2 px-3 font-bold rounded-lg text-[#4a5068] hover:text-white focus:outline-none transition-all";

        // 高亮顶层 (Layer 1) 的所有时间步
        for (let t = 1; t <= 7; t++) {
          const cell = document.getElementById(`g-1-${t}`);
          cell.className = "grid-cell w-9 h-9 border border-[#4ade80] bg-[#4ade80]/30 rounded-lg flex items-center justify-center text-[#4ade80] font-bold shadow-lg shadow-[#4ade80]/15";
        }

        alertBox.className = "mt-4 p-3 rounded-xl bg-[#4ade80]/10 border border-[#4ade80]/30 text-xs text-[#4ade80] leading-relaxed";
        alertBox.innerHTML = `
          <b>💡 output 的数学切片：</b><br>
          • <b>只取顶层（Layer 1）</b>，但保留<b>所有时间步</b>（t1 ~ t7）的特征 [2]。<br>
          • <b>它的物理含义是</b>：“随着句子的向前阅读，在顶层每一步产生的即时脑海印象”。常用于注意力机制 [2, 3]。
        `;
      } else {
        btnState.className = "flex-1 py-2 px-3 font-bold rounded-lg bg-[#fbbf24]/20 text-[#fbbf24] border border-[#fbbf24] focus:outline-none transition-all";
        btnOut.className = "flex-1 py-2 px-3 font-bold rounded-lg text-[#4a5068] hover:text-white focus:outline-none transition-all";

        // 高亮最后一个时刻 (t7) 的所有层
        for (let l = 0; l < 2; l++) {
          const cell = document.getElementById(`g-${l}-7`);
          cell.className = "grid-cell w-9 h-9 border border-[#fbbf24] bg-[#fbbf24]/30 rounded-lg flex items-center justify-center text-[#fbbf24] font-bold shadow-lg shadow-[#fbbf24]/15";
        }

        alertBox.className = "mt-4 p-3 rounded-xl bg-[#fbbf24]/10 border border-[#fbbf24]/30 text-xs text-[#fbbf24] leading-relaxed";
        alertBox.innerHTML = `
          <b>💡 state 的数学切片：</b><br>
          • <b>只取最后的时刻（t7）</b>，但保留<b>每一层（Layer 0, Layer 1）</b>的最终特征 [2]。<br>
          • <b>它的物理含义是</b>：“整句话读完时，大脑里每一层各自留存的终极压缩记忆”。直接作为初始状态丢给 Decoder [2]。
        `;
      }
    }

    // 初始化运行第一页
    switchTab('structure');
  </script>
</body>
</html>
"""

# 一键渲染到 Colab
from IPython.display import HTML
HTML(html_code)

尝试实现一个seq2seq-decoder

In [ ]:
class seq2seqDecoder(d2l.Decoder):
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers, dropout=0, **kwargs):
        super(seq2seqDecoder, self).__init__(**kwargs)
        # 词嵌入层
        self.embedding = nn.Embedding(vocab_size, embed_size)
        # GRU层，输入维度是词嵌入维度 + 隐藏状态维度（因为拼接了上下文向量）
        self.rnn = nn.GRU(embed_size + num_hiddens, num_hiddens, num_layers, dropout=dropout)
        # 全连接输出层，映射回词表大小
        self.dense = nn.Linear(num_hiddens, vocab_size)

    def init_state(self, enc_outputs, *args):
        # 返回编码器的最终隐藏状态作为初始状态
        return enc_outputs[1]

    def forward(self, x, state):
        # x 形状: (batch_size, num_steps) -> (num_steps, batch_size, embed_size)
        x = self.embedding(x).permute(1, 0, 2)
        # 取出编码器的最后一个隐藏状态并广播，使其形状变为 (num_steps, batch_size, num_hiddens); state[-1] 是最后一层的隐藏状态
        context = state[-1].repeat(x.shape[0], 1, 1)
        # 在特征维度上拼接输入和上下文向量
        x_and_context = torch.cat((x, context), 2)
        # 喂入 RNN
        output, state = self.rnn(x_and_context, state)
        # 映射到词表大小，形状变为 (num_steps, batch_size, vocab_size)
        output = self.dense(output).permute(1, 0, 2)
        return output, state

In [ ]:
# 实例化解码器：词表大小10，嵌入维度8，隐藏单元1（为了演示），GRU层数2
decoder = seq2seqDecoder(vocab_size=10, embed_size=8, num_hiddens=1, num_layers=2)

# 将模型设置为评估模式，关闭 dropout 等训练特有行为
decoder.eval()

# 准备模拟数据 X (batch_size=4, seq_len=7)
X = torch.zeros((4, 7), dtype=torch.long)

# 1. 初始化解码器状态：使用编码器对 X 的计算结果（通常是最终隐藏状态）作为解码器的起点
state = decoder.init_state(encoder(X))

# 2. 执行解码器前向传播：传入目标序列 X 和初始状态 state
# 返回 output (所有时间步的预测结果) 和 state (更新后的隐藏状态)
output, state = decoder(X, state)

# 查看输出张量的形状
# output 预期形状: (batch_size, seq_len, vocab_size) -> (4, 7, 10)
# state 预期形状: (num_layers, batch_size, num_hiddens) -> (2, 4, 1)
display(output.shape, state.shape)

torch.Size([4, 7, 10])

torch.Size([2, 4, 1])

可视化

In [ ]:
# @title
from IPython.display import HTML

html_code = r"""
<!DOCTYPE html>
<html>
<head>
  <script src="https://cdn.tailwindcss.com"></script>
  <style>
    @import url('https://fonts.googleapis.com/css2?family=Noto+Sans+SC:wght@400;500;700&display=swap');
    body {
      font-family: 'Noto Sans SC', sans-serif;
      background-color: #0a0c12;
      color: #dde1f0;
    }
    .tab-content { transition: all 0.3s ease; }
  </style>
</head>
<body class="p-4 md:p-6 min-h-screen flex items-center justify-center">

  <div class="w-full max-w-2xl bg-[#12151e] border border-[#1e2235] rounded-3xl p-6 shadow-2xl">

    <!-- 头部信息 -->
    <div class="text-center mb-6">
      <div class="text-[10px] text-[#2dd4bf] tracking-[4px] uppercase mb-1.5 font-bold">d2l · Seq2Seq</div>
      <h1 class="text-2xl font-extrabold text-white">Decoder 代码详解</h1>
      <div class="text-xs text-[#4a5068] mt-1.5">特征拼接（torch.cat）与自回归解码流转</div>
    </div>

    <!-- 1. 主导航 Tabs (4个) -->
    <div class="flex gap-2 p-1 bg-[#0a0c12] rounded-xl mb-6 flex-wrap text-xs">
      <button id="tab-btn-structure" onclick="switchTab('structure')" class="flex-1 py-2 px-3 font-bold rounded-lg bg-[#2dd4bf] text-white focus:outline-none transition-all">
        🏗️ 整体结构
      </button>
      <button id="tab-btn-dim" onclick="switchTab('dim')" class="flex-1 py-2 px-3 font-bold rounded-lg text-[#4a5068] hover:text-white focus:outline-none transition-all">
        📐 维度流转
      </button>
      <button id="tab-btn-concat" onclick="switchTab('concat')" class="flex-1 py-2 px-3 font-bold rounded-lg text-[#4a5068] hover:text-white focus:outline-none transition-all">
        🔗 特征拼接 (cat)
      </button>
      <button id="tab-btn-example" onclick="switchTab('example')" class="flex-1 py-2 px-3 font-bold rounded-lg text-[#4a5068] hover:text-white focus:outline-none transition-all">
        🔢 数值验证
      </button>
    </div>

    <!-- ========================================== -->
    <!-- Tab 1: 整体结构 (Structure) -->
    <!-- ========================================== -->
    <div id="tab-structure" class="tab-content space-y-4">
      <div class="bg-[#12151e] border border-[#1e2235] rounded-2xl p-5 space-y-3">
        <div class="text-sm font-bold text-[#2dd4bf]">整体结构：Embedding + GRU + Linear</div>
        <p class="text-xs text-[#4a5068] leading-relaxed">
          解码器的任务是：拿着编码器打包好的“整句终极记忆（Context）”，一步一步地将其“翻译、生成（Decode）”为目标语言 [2, 3]。
        </p>

        <div class="space-y-2.5 text-xs">
          <div class="bg-[#0a0c12] rounded-xl p-3 border border-[#22d3ee]/20">
            <div class="font-bold text-[#22d3ee] mb-1">① 目标词嵌入 nn.Embedding(vocab_size, embed_size)</div>
            <p class="text-[#4a5068] text-[10px]">将目标语言（如中文）的单词 ID，映射为 `embed_size` 维的特征向量 [2, 3]。</p>
          </div>
          <div class="bg-[#0a0c12] rounded-xl p-3 border border-[#8b7cf8]/20">
            <div class="font-bold text-[#8b7cf8] mb-1">② 门控循环单元 nn.GRU(embed_size + num_hiddens, num_hiddens)</div>
            <p class="text-[#4a5068] text-[10px]">核心计算单元。注意它的输入维度：它是词向量大小与隐藏状态大小的<b>相加之和</b>（因为拼接了上下文） [2]。</p>
          </div>
          <div class="bg-[#0a0c12] rounded-xl p-3 border border-[#4ade80]/20">
            <div class="font-bold text-[#4ade80] mb-1">③ 线性投射层 nn.Linear(num_hiddens, vocab_size)</div>
            <p class="text-[#4a5068] text-[10px]">将 GRU 吐出的隐藏特征，重新投影回目标语言的完整词表大小，输出每个候选词的 Logits 得分 [2]。</p>
          </div>
        </div>
      </div>

      <div class="bg-[#12151e] border border-[#1e2235] rounded-2xl p-5 space-y-2">
        <div class="text-xs font-bold text-[#fb923c]">💡 核心亮点：永不遗忘的“Context 拼接机制”</div>
        <p class="text-[11px] text-[#4a5068] leading-relaxed">
          传统的循环神经网络在读完长句子后，后面的时刻很容易遗忘最开始的信息。为了解决这个问题，我们在<b>每一个时间步 $t$</b>，都把<b>当前要翻译的词向量 $X_t$</b> 与<b>编码器压缩出的终极记忆向量 $Context$</b> 横向拼接（`torch.cat`）在一起，强行作为整体喂给 GRU [2, 3]。这确保了模型在翻译每一个词时，脑里都死死抓着原句的大意 [2, 3]。
        </p>
      </div>
    </div>

    <!-- ========================================== -->
    <!-- Tab 2: 维度流转 (Dim Flow) -->
    <!-- ========================================== -->
    <div id="tab-dim" class="tab-content space-y-4 hidden">
      <div class="bg-[#12151e] border border-[#1e2235] rounded-2xl p-5">
        <div class="text-sm font-bold text-[#dde1f0] mb-4">解码器数据前向传播维度变化图</div>

        <!-- Timeline Steps -->
        <div class="space-y-4">
          <!-- Step 1 -->
          <div class="flex">
            <div class="flex flex-col items-center w-8 mr-3">
              <span class="w-6 h-6 rounded-full bg-[#22d3ee]/10 border-2 border-[#22d3ee] flex items-center justify-center text-xs font-black text-[#22d3ee] z-10">1</span>
              <div class="w-0.5 bg-[#1e2235] flex-1 my-1"></div>
            </div>
            <div class="bg-[#0a0c12] border border-[#22d3ee]/20 rounded-xl p-3 flex-1 flex justify-between items-center">
              <div>
                <span class="text-xs font-bold text-[#22d3ee] block">1. 目标词输入 x (如中文词 id)</span>
                <span class="text-[10px] text-[#4a5068]">经过 Embedding 换轴后</span>
              </div>
              <code class="text-xs font-bold text-[#22d3ee] font-mono">(seq_len, batch, embed_size)</code>
            </div>
          </div>
          <!-- Step 2 -->
          <div class="flex">
            <div class="flex flex-col items-center w-8 mr-3">
              <span class="w-6 h-6 rounded-full bg-[#8b7cf8]/10 border-2 border-[#8b7cf8] flex items-center justify-center text-xs font-black text-[#8b7cf8] z-10">2</span>
              <div class="w-0.5 bg-[#1e2235] flex-1 my-1"></div>
            </div>
            <div class="bg-[#0a0c12] border border-[#8b7cf8]/20 rounded-xl p-3 flex-1 flex justify-between items-center">
              <div>
                <span class="text-xs font-bold text-[#8b7cf8] block">2. state[-1] 广播复制（Context）</span>
                <span class="text-[10px] text-[#4a5068]">编码器最后一层，复制 seq_len 份</span>
              </div>
              <code class="text-xs font-bold text-[#8b7cf8] font-mono">(seq_len, batch, num_hiddens)</code>
            </div>
          </div>
          <!-- Step 3 -->
          <div class="flex">
            <div class="flex flex-col items-center w-8 mr-3">
              <span class="w-6 h-6 rounded-full bg-[#fb923c]/10 border-2 border-[#fb923c] flex items-center justify-center text-xs font-black text-[#fb923c] z-10">3</span>
              <div class="w-0.5 bg-[#1e2235] flex-1 my-1"></div>
            </div>
            <div class="bg-[#0a0c12] border border-[#fb923c]/20 rounded-xl p-3 flex-1 flex justify-between items-center">
              <div>
                <span class="text-xs font-bold text-amber-400 block">3. torch.cat() 拼接特征</span>
                <span class="text-[10px] text-[#4a5068]">在最后一个维度（维度 2）上拼接</span>
              </div>
              <code class="text-xs font-bold text-amber-400 font-mono">(seq_len, batch, embed+hidden)</code>
            </div>
          </div>
          <!-- Step 4 -->
          <div class="flex">
            <div class="flex flex-col items-center w-8 mr-3">
              <span class="w-6 h-6 rounded-full bg-[#4ade80]/10 border-2 border-[#4ade80] flex items-center justify-center text-xs font-black text-[#4ade80] z-10">4</span>
              <div class="w-0.5 bg-[#1e2235] flex-1 my-1"></div>
            </div>
            <div class="bg-[#0a0c12] border border-[#4ade80]/20 rounded-xl p-3 flex-1 flex justify-between items-center">
              <div>
                <span class="text-xs font-bold text-[#4ade80] block">4. GRU 循环层最终计算</span>
                <span class="text-[10px] text-[#4a5068]">输出顶层在所有时间步的特征</span>
              </div>
              <code class="text-xs font-bold text-[#4ade80] font-mono">(seq_len, batch, num_hiddens)</code>
            </div>
          </div>
          <!-- Step 5 -->
          <div class="flex">
            <div class="flex flex-col items-center w-8 mr-3">
              <span class="w-6 h-6 rounded-full bg-[#f472b6]/10 border-2 border-[#f472b6] flex items-center justify-center text-xs font-black text-[#f472b6] z-10">5</span>
            </div>
            <div class="bg-[#0a0c12] border border-[#f472b6]/20 rounded-xl p-3 flex-1 flex justify-between items-center">
              <div>
                <span class="text-xs font-bold text-pink-400 block">5. self.dense 线性投影 ──> 换轴</span>
                <span class="text-[10px] text-[#4a5068]">映射回目标语言词表大小，换回 Batch-first</span>
              </div>
              <code class="text-xs font-bold text-pink-400 font-mono">(batch, seq_len, vocab_size)</code>
            </div>
          </div>
        </div>
      </div>
    </div>

    <!-- ========================================== -->
    <!-- Tab 3: 特征拼接 (Concat) -->
    <!-- ========================================== -->
    <div id="tab-concat" class="tab-content space-y-4 hidden">
      <div class="bg-[#12151e] border border-[#1e2235] rounded-2xl p-5">
        <div class="text-sm font-bold text-amber-400 mb-2">图形化拆解：`torch.cat((x, context), 2)` 的拼接魔术</div>
        <p class="text-xs text-[#4a5068] leading-relaxed mb-6">
          拼接发生在最后一个维度（即维度 2，特征维度）上。我们取其中<b>一个具体位置的单词</b>来看：
        </p>

        <!-- Concat Interactive Demo -->
        <div class="flex flex-col items-center justify-center gap-4 bg-[#0a0c12] p-6 rounded-2xl border border-[#1e2235] mb-4">
          <!-- Vector X (Embed) -->
          <div class="flex items-center gap-2">
            <span class="w-24 text-right text-xs text-[#22d3ee] font-bold">当前词向量 x :</span>
            <div class="flex gap-1">
              <div class="w-7 h-7 bg-[#22d3ee]/20 border border-[#22d3ee] rounded-md flex items-center justify-center text-[10px] text-[#22d3ee] font-mono">e1</div>
              <div class="w-7 h-7 bg-[#22d3ee]/20 border border-[#22d3ee] rounded-md flex items-center justify-center text-[10px] text-[#22d3ee] font-mono">e2</div>
              <div class="w-7 h-7 bg-[#22d3ee]/20 border border-[#22d3ee] rounded-md flex items-center justify-center text-[10px] text-[#22d3ee] font-mono">e3</div>
              <div class="w-7 h-7 bg-[#22d3ee]/20 border border-[#22d3ee] rounded-md flex items-center justify-center text-[10px] text-[#22d3ee] font-mono">e4</div>
              <div class="w-7 h-7 bg-[#22d3ee]/20 border border-[#22d3ee] rounded-md flex items-center justify-center text-[10px] text-[#22d3ee] font-mono">e5</div>
              <div class="w-7 h-7 bg-[#22d3ee]/20 border border-[#22d3ee] rounded-md flex items-center justify-center text-[10px] text-[#22d3ee] font-mono">e6</div>
            </div>
            <span class="text-[10px] text-[#4a5068] font-mono">(6 维)</span>
          </div>

          <span class="text-lg text-[#4a5068] font-bold">+ (横向拼接)</span>

          <!-- Vector Context -->
          <div class="flex items-center gap-2">
            <span class="w-24 text-right text-xs text-[#8b7cf8] font-bold">整句记忆 context :</span>
            <div class="flex gap-1">
              <div class="w-7 h-7 bg-[#8b7cf8]/20 border border-[#8b7cf8] rounded-md flex items-center justify-center text-[10px] text-[#8b7cf8] font-mono">h1</div>
              <div class="w-7 h-7 bg-[#8b7cf8]/20 border border-[#8b7cf8] rounded-md flex items-center justify-center text-[10px] text-[#8b7cf8] font-mono">h2</div>
              <div class="w-7 h-7 bg-[#8b7cf8]/20 border border-[#8b7cf8] rounded-md flex items-center justify-center text-[10px] text-[#8b7cf8] font-mono">h3</div>
              <div class="w-7 h-7 bg-[#8b7cf8]/20 border border-[#8b7cf8] rounded-md flex items-center justify-center text-[10px] text-[#8b7cf8] font-mono">h4</div>
              <div class="w-7 h-7 bg-[#8b7cf8]/20 border border-[#8b7cf8] rounded-md flex items-center justify-center text-[10px] text-[#8b7cf8] font-mono">h5</div>
              <div class="w-7 h-7 bg-[#8b7cf8]/20 border border-[#8b7cf8] rounded-md flex items-center justify-center text-[10px] text-[#8b7cf8] font-mono">h6</div>
              <div class="w-7 h-7 bg-[#8b7cf8]/20 border border-[#8b7cf8] rounded-md flex items-center justify-center text-[10px] text-[#8b7cf8] font-mono">h7</div>
              <div class="w-7 h-7 bg-[#8b7cf8]/20 border border-[#8b7cf8] rounded-md flex items-center justify-center text-[10px] text-[#8b7cf8] font-mono">h8</div>
            </div>
            <span class="text-[10px] text-[#4a5068] font-mono">(8 维)</span>
          </div>

          <span class="text-lg text-emerald-400 font-bold">➔ (cat 运行后)</span>

          <!-- Vector Concatenated -->
          <div class="flex items-center gap-2">
            <span class="w-24 text-right text-xs text-emerald-400 font-bold">拼接后的输入 :</span>
            <div class="flex gap-1 flex-wrap max-w-sm justify-center">
              <div class="w-7 h-7 bg-[#22d3ee]/20 border border-[#22d3ee] rounded-md flex items-center justify-center text-[10px] text-[#22d3ee] font-mono">e1</div>
              <div class="w-7 h-7 bg-[#22d3ee]/20 border border-[#22d3ee] rounded-md flex items-center justify-center text-[10px] text-[#22d3ee] font-mono">e2</div>
              <div class="w-7 h-7 bg-[#22d3ee]/20 border border-[#22d3ee] rounded-md flex items-center justify-center text-[10px] text-[#22d3ee] font-mono">e3</div>
              <div class="w-7 h-7 bg-[#22d3ee]/20 border border-[#22d3ee] rounded-md flex items-center justify-center text-[10px] text-[#22d3ee] font-mono">e4</div>
              <div class="w-7 h-7 bg-[#22d3ee]/20 border border-[#22d3ee] rounded-md flex items-center justify-center text-[10px] text-[#22d3ee] font-mono">e5</div>
              <div class="w-7 h-7 bg-[#22d3ee]/20 border border-[#22d3ee] rounded-md flex items-center justify-center text-[10px] text-[#22d3ee] font-mono">e6</div>

              <div class="w-7 h-7 bg-[#8b7cf8]/20 border border-[#8b7cf8] rounded-md flex items-center justify-center text-[10px] text-[#8b7cf8] font-mono">h1</div>
              <div class="w-7 h-7 bg-[#8b7cf8]/20 border border-[#8b7cf8] rounded-md flex items-center justify-center text-[10px] text-[#8b7cf8] font-mono">h2</div>
              <div class="w-7 h-7 bg-[#8b7cf8]/20 border border-[#8b7cf8] rounded-md flex items-center justify-center text-[10px] text-[#8b7cf8] font-mono">h3</div>
              <div class="w-7 h-7 bg-[#8b7cf8]/20 border border-[#8b7cf8] rounded-md flex items-center justify-center text-[10px] text-[#8b7cf8] font-mono">h4</div>
              <div class="w-7 h-7 bg-[#8b7cf8]/20 border border-[#8b7cf8] rounded-md flex items-center justify-center text-[10px] text-[#8b7cf8] font-mono">h5</div>
              <div class="w-7 h-7 bg-[#8b7cf8]/20 border border-[#8b7cf8] rounded-md flex items-center justify-center text-[10px] text-[#8b7cf8] font-mono">h6</div>
              <div class="w-7 h-7 bg-[#8b7cf8]/20 border border-[#8b7cf8] rounded-md flex items-center justify-center text-[10px] text-[#8b7cf8] font-mono">h7</div>
              <div class="w-7 h-7 bg-[#8b7cf8]/20 border border-[#8b7cf8] rounded-md flex items-center justify-center text-[10px] text-[#8b7cf8] font-mono">h8</div>
            </div>
            <span class="text-[10px] text-[#4a5068] font-mono">(14 维)</span>
          </div>
        </div>

        <p class="text-[11px] text-[#4a5068] leading-relaxed text-center">
          在最后一个特征维度上拼接，长度直接相加：$6（embed\_size） + 8（num\_hiddens） = 14$ [2]。<br>
          因此，我们定义解码器的 GRU 输入大小必须是 **`14`**，才能完美接收并大口吞下这个拼接后的巨大向量 [2]！
        </p>
      </div>
    </div>

    <!-- ========================================== -->
    <!-- Tab 4: 例子验证 (Example) -->
    <!-- ========================================== -->
    <div id="tab-example" class="tab-content space-y-4 hidden">
      <div class="bg-[#12151e] border border-[#1e2235] rounded-2xl p-5 space-y-4">
        <div class="text-sm font-bold text-[#dde1f0]">代入实例中的真实参数</div>
        <div class="grid grid-cols-2 gap-3 text-[10px]">
          <div class="bg-[#0a0c12] border border-[#22d3ee]/20 p-2.5 rounded-xl">
            <span class="bg-[#22d3ee]/10 text-[#22d3ee] px-1.5 py-0.5 rounded border border-[#22d3ee]/20 font-bold block mb-1">vocab_size=10</span>
            <span class="text-[#4a5068]">目标词表大小为 10。</span>
          </div>
          <div class="bg-[#0a0c12] border border-purple-500/20 p-2.5 rounded-xl">
            <span class="bg-purple-500/10 text-purple-400 px-1.5 py-0.5 rounded border border-purple-500/20 font-bold block mb-1">embed_size=6</span>
            <span class="text-[#4a5068]">目标词向量维度为 6。</span>
          </div>
          <div class="bg-[#0a0c12] border border-orange-500/20 p-2.5 rounded-xl">
            <span class="bg-orange-500/10 text-orange-400 px-1.5 py-0.5 rounded border border-orange-500/20 font-bold block mb-1">num_hiddens=8</span>
            <span class="text-[#4a5068]">隐藏状态（记忆大小）为 8。</span>
          </div>
          <div class="bg-[#0a0c12] border border-emerald-500/20 p-2.5 rounded-xl">
            <span class="bg-emerald-500/10 text-emerald-400 px-1.5 py-0.5 rounded border border-emerald-500/20 font-bold block mb-1">num_layers=2</span>
            <span class="text-[#4a5068]">纵向叠 2 层 GRU 进行处理 [2]。</span>
          </div>
          <div class="bg-[#0a0c12] border border-yellow-500/20 p-2.5 rounded-xl">
            <span class="bg-yellow-500/10 text-yellow-400 px-1.5 py-0.5 rounded border border-yellow-500/20 font-bold block mb-1">batch_size=4</span>
            <span class="text-[#4a5068]">一次性并行处理 4 个句子 [3]。</span>
          </div>
          <div class="bg-[#0a0c12] border border-pink-500/20 p-2.5 rounded-xl">
            <span class="bg-pink-500/10 text-pink-400 px-1.5 py-0.5 rounded border border-pink-500/20 font-bold block mb-1">seq_len=7</span>
            <span class="text-[#4a5068]">解码的中文句子有 7 个字长。</span>
          </div>
        </div>
      </div>

      <!-- Verification box -->
      <div class="bg-[#12151e] border border-[#1e2235] rounded-2xl p-5 space-y-3">
        <div class="text-sm font-bold text-[#2dd4bf]">解码器最终输出物理解析</div>
        <pre class="bg-[#0a0c12] text-xs font-mono p-4 rounded-xl leading-relaxed text-[#dde1f0]">
output.shape = (<span class="text-yellow-400">4</span>, <span class="text-pink-400">7</span>, <span class="text-[#2dd4bf]">10</span>)
                ↑batch  ↑seq  ↑vocab_size

# 物理意义：
在预测阶段，解码器为这 4 个句子中的每一个单词，
都并行输出一个 10 分类的原始得分向量（Logits），
用于评估哪个汉字最应该作为这一步的翻译答案 [2, 3]！
        </pre>
      </div>
    </div>

  </div>

  <script>
    // 1. 主 Tab 切换逻辑
    const tabContents = ['structure', 'dim', 'concat', 'example'];
    function switchTab(activeId) {
      tabContents.forEach(id => {
        const btn = document.getElementById(`tab-btn-${id}`);
        const content = document.getElementById(`tab-${id}`);
        if (id === activeId) {
          btn.className = "flex-1 py-2 px-3 font-bold rounded-lg bg-[#2dd4bf] text-white focus:outline-none transition-all";
          content.classList.remove('hidden');
        } else {
          btn.className = "flex-1 py-2 px-3 font-bold rounded-lg text-[#4a5068] hover:text-white focus:outline-none transition-all";
          content.classList.add('hidden');
        }
      });
    }

    // 初始化运行第一页
    switchTab('structure');
  </script>
</body>
</html>
"""

# 一键渲染到 Colab 单元格
from IPython.display import HTML
HTML(html_code)

机器翻译举例：

数据与模型配置对齐
在开始计算前，我们先为本次翻译任务确立基础的数据字典和参数尺寸：

1. 英语源词表（Encoder 端）
* **词表大小 `vocab_size_enc` = 10**（这里假定英文词表只有10个词）：
  * `"<pad>"` $\to$ `0`
  * `"I"` $\to$ `1`
  * `"love"` $\to$ `2`
  * `"AI"` $\to$ `3`
  * ……
* **英文输入**：`"I love AI"` $\longrightarrow$ 分词映射为整数列表：`[1, 2, 3]` （长度 $T_{enc} = 3$）

2. 中文目标词表（Decoder 端）
* **词表大小 `vocab_size_dec` = 8**（这里假定中文词表只有8个词）：
  * `"<bos>"` $\to$ `0`（Start Token，句子开始标记）
  * `"我"` $\to$ `1`
  * `"爱"` $\to$ `2`
  * `"AI"` $\to$ `3`
  * `"<eos>"` $\to$ `4`（End Token，句子结束标记）
  * ……
* **中文输出目标**：`["我", "爱", "AI", "<eos>"]`

3. 模型超参数配置
* 一次只翻译一个句子：**`batch_size` ($B$) = 1**
* 词嵌入向量维度：**`embed_size` ($C$) = 4**（每个单词用 4 维表示）
* GRU 记忆体维度：**`num_hiddens` ($H$) = 6**（隐藏状态大小为 6）
* 循环层数：**`num_layers` ($L$) = 1**（1层 GRU）


In [ ]:
# @title
from IPython.display import HTML

html_code = r"""
<!DOCTYPE html>
<html>
<head>
  <script src="https://cdn.tailwindcss.com"></script>
  <style>
    @import url('https://fonts.googleapis.com/css2?family=Noto+Sans+SC:wght@400;500;700&display=swap');
    body {
      font-family: 'Noto Sans SC', sans-serif;
      background-color: #0a0c12;
      color: #dde1f0;
    }
    .tab-content { transition: all 0.3s ease; }
    .step-card { transition: all 0.2s ease; }
  </style>
</head>
<body class="p-4 md:p-6 min-h-screen flex items-center justify-center">

  <div class="w-full max-w-2xl bg-[#12151e] border border-[#1e2235] rounded-3xl p-6 shadow-2xl">

    <!-- 标题栏 -->
    <div class="text-center mb-6">
      <div class="text-[10px] text-[#8b7cf8] tracking-[4px] uppercase mb-1 font-bold">Seq2Seq Real Case Study</div>
      <h1 class="text-xl font-extrabold text-white">英翻中："I love AI" ➔ "我爱AI"</h1>
      <div class="text-xs text-[#4a5068] mt-1">手把手数据前向流转 & 状态生命周期追踪</div>
    </div>

    <!-- 1. 主导航 Tabs -->
    <div class="flex gap-2 p-1 bg-[#0a0c12] rounded-xl mb-6 text-xs">
      <button id="btn-tab-flow" onclick="changeTab('flow')" class="flex-1 py-2 px-3 font-bold rounded-lg bg-[#8b7cf8] text-white focus:outline-none transition-all">
        🗺️ 宏观流程
      </button>
      <button id="btn-tab-encoder" onclick="changeTab('encoder')" class="flex-1 py-2 px-3 font-bold rounded-lg text-[#4a5068] hover:text-white focus:outline-none transition-all">
        🏋️‍♂️ 编码器端 (Encoder)
      </button>
      <button id="btn-tab-decoder" onclick="changeTab('decoder')" class="flex-1 py-2 px-3 font-bold rounded-lg text-[#4a5068] hover:text-white focus:outline-none transition-all">
        ✨ 解码生成 (Decoder)
      </button>
    </div>

    <!-- ========================================== -->
    <!-- Tab 1: 宏观流程 (Flow) -->
    <!-- ========================================== -->
    <div id="tab-flow" class="tab-content space-y-4">
      <div class="bg-[#0a0c12] border border-[#1e2235] rounded-2xl p-5 space-y-4">
        <div class="text-sm font-bold text-[#8b7cf8]">Seq2Seq 翻译宏观大盘</div>

        <!-- Flow Diagram -->
        <div class="flex flex-col items-center justify-center gap-2 py-4 bg-[#12151e] rounded-xl border border-[#1e2235]">
          <!-- Encoder -->
          <div class="text-center">
            <span class="text-[10px] text-[#4a5068] block mb-1">输入英文序列 (Seq_len=3)</span>
            <div class="flex gap-1.5 justify-center">
              <span class="bg-[#22d3ee]/10 text-[#22d3ee] border border-[#22d3ee]/30 px-3 py-1 rounded-lg text-xs font-bold font-mono">I (1)</span>
              <span class="bg-[#22d3ee]/10 text-[#22d3ee] border border-[#22d3ee]/30 px-3 py-1 rounded-lg text-xs font-bold font-mono">love (2)</span>
              <span class="bg-[#22d3ee]/10 text-[#22d3ee] border border-[#22d3ee]/30 px-3 py-1 rounded-lg text-xs font-bold font-mono">AI (3)</span>
            </div>
          </div>

          <span class="text-lg text-[#4a5068] font-bold">⬇ (进入 Encoder)</span>

          <!-- Context Vector -->
          <div class="bg-[#fbbf24]/10 text-[#fbbf24] border border-[#fbbf24]/40 px-4 py-2.5 rounded-xl text-center max-w-sm">
            <span class="text-[10px] uppercase block mb-1 font-bold">编码器最终隐藏状态 state (Context 向量)</span>
            <code class="text-xs font-mono font-bold">[1, 1, 6] ──> [h1, h2, h3, h4, h5, h6]</code>
            <p class="text-[9px] text-[#4a5068] mt-1">它压缩了整句 "I love AI" 的精髓语义 [2, 3]。</p>
          </div>

          <span class="text-lg text-[#4a5068] font-bold">⬇ (作为初始状态喂给 Decoder)</span>

          <!-- Decoder -->
          <div class="text-center">
            <span class="text-[10px] text-[#4a5068] block mb-1">自回归逐步解码 (Auto-regressive Generation)</span>
            <div class="flex gap-1.5 justify-center">
              <span class="bg-[#4ade80]/15 text-[#4ade80] border border-[#4ade80]/30 px-2.5 py-1 rounded-lg text-xs font-bold">我 (1)</span>
              <span class="bg-[#4ade80]/15 text-[#4ade80] border border-[#4ade80]/30 px-2.5 py-1 rounded-lg text-xs font-bold">爱 (2)</span>
              <span class="bg-[#4ade80]/15 text-[#4ade80] border border-[#4ade80]/30 px-2.5 py-1 rounded-lg text-xs font-bold">AI (3)</span>
              <span class="bg-rose-500/15 text-rose-400 border border-rose-500/30 px-2.5 py-1 rounded-lg text-xs font-bold">&lt;eos&gt; (4)</span>
            </div>
          </div>
        </div>
      </div>
    </div>

    <!-- ========================================== -->
    <!-- Tab 2: 编码器单步跟踪 (Encoder) -->
    <!-- ========================================== -->
    <div id="tab-encoder" class="tab-content space-y-4 hidden">
      <div class="bg-[#0a0c12] border border-[#1e2235] rounded-2xl p-5 space-y-4">
        <div class="text-sm font-bold text-[#22d3ee]">Encoder 数据流与形状变化</div>

        <div class="space-y-3 font-mono text-xs">
          <!-- Step 1 -->
          <div class="p-3 bg-[#12151e] rounded-xl border border-[#1e2235]">
            <div class="flex justify-between items-center mb-1">
              <span class="text-[#22d3ee] font-bold">1. 原始输入 x_enc</span>
              <span class="text-[#22d3ee]">shape = (1, 3)</span>
            </div>
            <p class="text-[10px] text-[#4a5068] mb-2 font-sans">代表 1 个句子，内含 3 个英文单词 ID。</p>
            <code class="text-slate-300">tensor([[1, 2, 3]])  # ["I", "love", "AI"]</code>
          </div>
          <!-- Step 2 -->
          <div class="p-3 bg-[#12151e] rounded-xl border border-[#1e2235]">
            <div class="flex justify-between items-center mb-1">
              <span class="text-[#8b7cf8] font-bold">2. self.embedding(x_enc)</span>
              <span class="text-[#8b7cf8]">shape = (1, 3, 4)</span>
            </div>
            <p class="text-[10px] text-[#4a5068] mb-2 font-sans">每个单词转换为 4 维特征向量 [2, 3]。</p>
            <code class="text-slate-400">[[ [e1,e2,e3,e4], [e1,e2,e3,e4], [e1,e2,e3,e4] ]]</code>
          </div>
          <!-- Step 3 -->
          <div class="p-3 bg-[#12121e] rounded-xl border border-[#1e2235]">
            <div class="flex justify-between items-center mb-1">
              <span class="text-amber-400 font-bold">3. x_enc.permute(1, 0, 2) 换轴</span>
              <span class="text-amber-400">shape = (3, 1, 4)</span>
            </div>
            <p class="text-[10px] text-[#4a5068] mb-1 font-sans">时间步（Seq_len）挪到第一维度，准备喂入 GRU [2]。</p>
          </div>
          <!-- Step 4 -->
          <div class="p-3 bg-[#12151e] rounded-xl border border-[#1e2235]">
            <div class="flex justify-between items-center mb-1">
              <span class="text-[#4ade80] font-bold">4. output, state = self.rnn(x_enc)</span>
              <span class="text-[#4ade80]">state.shape = (1, 1, 6)</span>
            </div>
            <p class="text-[10px] text-[#4a5068] mb-2 font-sans">GRU（隐藏层大小为 6）读完最后一字时，吐出终极记忆向量 [2, 3]。</p>
            <code class="text-emerald-400">state = tensor([[[h1, h2, h3, h4, h5, h6]]])</code>
          </div>
        </div>
      </div>
    </div>

    <!-- ========================================== -->
    <!-- Tab 3: 解码器自回归生成 (Decoder) -->
    <!-- ========================================== -->
    <div id="tab-decoder" class="tab-content space-y-4 hidden">
      <div class="bg-[#0a0c12] border border-[#1e2235] rounded-2xl p-5 space-y-4">
        <div class="text-sm font-bold text-[#4ade80] mb-2">自回归解码生成步骤模拟</div>

        <!-- Decoder Sub-steps (4 steps) -->
        <div class="flex gap-1.5 p-0.5 bg-[#12151e] rounded-lg text-[10px] font-bold">
          <button id="btn-d1" onclick="showDecStep(1)" class="flex-1 py-1 px-1 rounded bg-[#4ade80]/20 text-[#4ade80] focus:outline-none transition-all">第一步 ("我")</button>
          <button id="btn-d2" onclick="showDecStep(2)" class="flex-1 py-1 px-1 rounded text-[#4a5068] hover:text-white focus:outline-none transition-all">第二步 ("爱")</button>
          <button id="btn-d3" onclick="showDecStep(3)" class="flex-1 py-1 px-1 rounded text-[#4a5068] hover:text-white focus:outline-none transition-all">第三步 ("AI")</button>
          <button id="btn-d4" onclick="showDecStep(4)" class="flex-1 py-1 px-1 rounded text-[#4a5068] hover:text-white focus:outline-none transition-all">第四步 (结束)</button>
        </div>

        <!-- Dynamic Step Content -->
        <div class="bg-[#12151e] p-4 rounded-xl border border-slate-800 space-y-4 text-xs">
          <!-- Trace Diagram -->
          <div>
            <span class="text-[10px] text-[#4a5068] block mb-2 uppercase font-bold">数据输入与拼接流图</span>
            <div class="flex justify-center items-center gap-2 font-mono text-[10px] py-1 bg-[#0a0c12] rounded-xl border border-slate-900">
              <div class="text-center">
                <span class="text-[8px] text-[#4a5068] block mb-1">当前输入 (x_dec)</span>
                <span id="dec-input-val" class="px-2 py-1 rounded bg-cyan-950/50 border border-cyan-800 text-cyan-300 font-bold">"&lt;bos&gt;" (0)</span>
              </div>
              <span class="text-[#4a5068] font-bold">+</span>
              <div class="text-center">
                <span class="text-[8px] text-[#4a5068] block mb-1">重复的 Context</span>
                <span class="px-2 py-1 rounded bg-purple-950/50 border border-purple-800 text-purple-300 font-bold">"I love AI" (6维)</span>
              </div>
              <span class="text-[#4a5068] font-bold">➔</span>
              <div class="text-center">
                <span class="text-[8px] text-emerald-400 block mb-1">拼接后输入 (10维)</span>
                <span class="px-2 py-1 rounded bg-emerald-950/50 border border-emerald-800 text-emerald-300 font-bold">cat [4 + 6]</span>
              </div>
            </div>
          </div>

          <!-- Predictions -->
          <div>
            <span class="text-[10px] text-[#4a5068] block mb-1.5 uppercase font-bold">模型预测输出与决策</span>
            <div class="p-3 bg-[#0a0c12] rounded-xl border border-slate-900 font-mono space-y-1 text-[#dde1f0]">
              <div class="flex justify-between">
                <span>模型输出 Logits 形状:</span>
                <code class="text-pink-400 font-bold">torch.Size([1, 1, 8])</code>
              </div>
              <div class="flex justify-between">
                <span>最高分对应的汉字 ID:</span>
                <code id="dec-pred-id" class="text-emerald-400 font-bold">1</code>
              </div>
              <div class="flex justify-between">
                <span>一键恢复出的预测汉字:</span>
                <code id="dec-pred-word" class="text-emerald-400 font-bold">"我"</code>
              </div>
            </div>
          </div>

          <!-- Explainer -->
          <div class="p-3 bg-slate-900 border border-slate-800/80 rounded-xl leading-relaxed text-[#8fa3c7]">
            <span class="text-xs font-bold text-amber-400 block mb-1">单步执行逻辑解析</span>
            <p id="dec-explainer">
              计算加载中...
            </p>
          </div>
        </div>
      </div>
    </div>

  </div>

  <script>
    // 1. 主 Tab 切换逻辑
    const mainTabs = ['flow', 'encoder', 'decoder'];
    function changeTab(activeId) {
      mainTabs.forEach(id => {
        const btn = document.getElementById(`btn-tab-${id}`);
        const content = document.getElementById(`tab-${id}`);
        if (id === activeId) {
          btn.className = "flex-1 py-2 px-3 font-bold rounded-lg bg-[#8b7cf8] text-white focus:outline-none transition-all";
          content.classList.remove('hidden');
        } else {
          btn.className = "flex-1 py-2 px-3 font-bold rounded-lg text-[#4a5068] hover:text-white focus:outline-none transition-all";
          content.classList.add('hidden');
        }
      });
      // 触发一下，确保第三个tab打开时样式和内容初始化正确
      if (activeId === 'decoder') {
        showDecStep(1);
      }
    }

    // 2. Decoder 步骤切换配置
    const decStepsConfig = {
      1: {
        input: `"&lt;bos&gt;" (0)`,
        predId: `1`,
        predWord: `"我"`,
        explainer: `
          模型接收开局起始符 <b>"&lt;bos&gt;"</b> 开启翻译。我们把这个词对应的 Embedding 向量（4维），与编码器送过来的<b>整句记忆 Context（6维）</b>在特征轴上拼在一起，组成 10 维输入喂给 GRU [2]。<br>
          输出层 ` + "`self.dense`" + ` 将其投影为 8 维词表得分，其中 <b>"我"（ID 1）</b> 拿到了最高分 [2]！
        `
      },
      2: {
        input: `"我" (1)`,
        predId: `2`,
        predWord: `"爱"`,
        explainer: `
          <b>自回归滚雪球开始</b>：第一步预测出来的 <b>"我"</b>（ID 1）会被当作新的输入，继续喂入解码器。为了不忘本，它依然要和<b>整句记忆 Context（6维）</b>拼在一起共同喂入 GRU [2, 3]。<br>
          这一次，GRU 记着刚才它已经吐过“我”字了。结合语感，输出层预测出下一个概率最高的汉字是 <b>"爱"（ID 2）</b> [3]。
        `
      },
      3: {
        input: `"爱" (2)`,
        predId: `3`,
        predWord: `"AI"`,
        explainer: `
          将上一步预测出来的 <b>"爱"</b>（ID 2）再次喂回给模型作为输入，同时拼接<b>同一个 Context（6维）</b> [3]。<br>
          模型结合前面的上文“我、爱”，在输出的 8 维词表里进行概率检索，判定下一个最合理的汉字是 <b>"AI"（ID 3）</b> [3]。
        `
      },
      4: {
        input: `"AI" (3)`,
        predId: `4`,
        predWord: `"&lt;eos&gt;" (结束标志)`,
        explainer: `
          将上一步预测出来的 <b>"AI"</b>（ID 3）喂回。模型看清了前文“我、爱、AI”，并且对比了英文大意，判定<b>翻译已经完美结束了</b>！<br>
          输出层为结束标志符 <b>"&lt;eos&gt;"（ID 4）</b> 打出了最高分。当解码器探测到预测结果为结束符时，宣告前向翻译循环正式圆满结束 [3]！
        `
      }
    };

    function showDecStep(stepNum) {
      // 1. 更新按钮样式
      for (let i = 1; i <= 4; i++) {
        const btn = document.getElementById(`btn-d${i}`);
        if (i === stepNum) {
          btn.className = "flex-1 py-1 px-1 rounded bg-[#4ade80]/20 text-[#4ade80] font-bold focus:outline-none transition-all";
        } else {
          btn.className = "flex-1 py-1 px-1 rounded text-[#4a5068] hover:text-white focus:outline-none transition-all";
        }
      }

      // 2. 注入动态内容
      const info = decStepsConfig[stepNum];
      document.getElementById('dec-input-val').innerHTML = info.input;
      document.getElementById('dec-pred-id').innerHTML = info.predId;
      document.getElementById('dec-pred-word').innerHTML = info.predWord;
      document.getElementById('dec-explainer').innerHTML = info.explainer;
    }

    // 初始化运行
    changeTab('flow');
  </script>
</body>
</html>
"""

# 一键渲染到 Colab 单元格
from IPython.display import HTML
HTML(html_code)

填充掩码（padding）

In [ ]:
def sequence_mask(X, valid_len, value=0):
    """在序列中屏蔽不相关的项"""
    # 1. 获取序列的最大长度 (列数)
    maxlen = X.size(1)

    # 2. 核心逻辑：广播比较 (Broadcasting)
    # torch.arange(maxlen) 生成 [0, 1, 2, ..., maxlen-1]
    # [None, :] 将其变为行向量 (1, maxlen)
    # valid_len[:, None] 将其变为列向量 (batch_size, 1)
    # 比较后利用广播机制生成 (batch_size, maxlen) 的布尔掩码矩阵
    mask = torch.arange((maxlen), dtype=torch.float32,
                         device=X.device)[None, :] < valid_len[:, None]

    # 3. 取反掩码 (~mask)：将有效长度之后的位置(False)替换为指定值(通常是0)
    X[~mask] = value
    return X

# 测试代码
X = torch.tensor([[1, 2, 3], [4, 5, 6]])
valid_len = torch.tensor([1, 2])
sequence_mask(X, valid_len)

tensor([[1, 0, 0],
        [4, 5, 0]])

In [ ]:
# @title 🎭 序列掩码 (Sequence Masking) 深度逻辑拆解 {display-mode: "form"}
import json
import torch
from IPython.display import HTML
from google.colab import output

# 定义汇报JS错误的后端回调
def _report_js_error(message):
    print(f"JavaScript Error: {message}")
output.register_callback('report_js_error', _report_js_error)

# 准备展示数据
mask_data = {
    "tensor_x": [[1, 2, 3], [4, 5, 6]],
    "valid_lens": [1, 2],
    "maxlen": 3
}

dashboard_html = r"""
<!DOCTYPE html>
<html>
<head>
  <script src="https://cdn.tailwindcss.com"></script>
  <style>
    :root {
      --bg: #0a0c12; --card: #12151e; --border: #1e2235;
      --cyan: #22d3ee; --purple: #8b7cf8; --amber: #fbbf24;
    }
    body { background: var(--bg); color: #dde1f0; font-family: 'Inter', system-ui, sans-serif; }
    .step-box { background: var(--card); border: 1px solid var(--border); border-radius: 16px; padding: 20px; transition: transform 0.2s; }
    .tensor-cell { width: 42px; height: 42px; display: flex; align-items: center; justify-content: center;
                   border-radius: 8px; font-family: monospace; font-weight: bold; font-size: 14px; }
    .mask-true { background: rgba(34, 211, 238, 0.15); border: 1px solid var(--cyan); color: var(--cyan); }
    .mask-false { background: rgba(244, 63, 94, 0.1); border: 1px dashed #f43f5e; color: #f43f5e; opacity: 0.5; }
    .val-active { background: #1a1d2a; border: 1px solid var(--border); }
    .val-muted { background: #0a0c12; border: 1px dashed #4a5068; color: #4a5068; text-decoration: line-through; }
    .code-hl { color: var(--cyan); font-family: monospace; font-weight: bold; }
  </style>
</head>
<body class="p-6">
  <div class="max-w-3xl mx-auto space-y-6">

    <!-- Header -->
    <div class="text-center">
      <h1 class="text-2xl font-black text-white mb-2 tracking-tight">sequence_mask 算子可视化</h1>
      <p class="text-sm text-slate-500">解构 PyTorch 广播机制与布尔索引的协作过程</p>
    </div>

    <!-- Step 1: Broadcasting -->
    <div class="step-box shadow-xl">
      <div class="flex items-center gap-2 mb-4">
        <span class="px-2 py-0.5 bg-cyan-500/20 text-cyan-400 text-[10px] font-bold rounded-full">STEP 1</span>
        <h3 class="text-sm font-bold text-slate-200">广播比较逻辑生成 Mask</h3>
      </div>
      <div class="bg-[#0a0c12] p-4 rounded-xl mb-4 font-mono text-xs leading-relaxed">
        <span class="text-slate-500"># arange(3) < valid_len</span><br>
        <span class="code-hl">[[0, 1, 2]]</span> <span class="text-slate-500"># (1, 3)</span><br>
        <span class="text-amber-400">&lt;</span><br>
        <span class="code-hl">[[1], [2]]</span> <span class="text-slate-500"># (2, 1)</span>
      </div>
      <div id="mask-grid-container" class="space-y-3"></div>
    </div>

    <!-- Step 2: Boolean Indexing -->
    <div class="step-box shadow-xl">
      <div class="flex items-center gap-2 mb-4">
        <span class="px-2 py-0.5 bg-purple-500/20 text-purple-400 text-[10px] font-bold rounded-full">STEP 2</span>
        <h3 class="text-sm font-bold text-slate-200">应用掩码：X[~mask] = 0</h3>
      </div>
      <p class="text-[11px] text-slate-500 mb-4">取反掩码（~mask）会选中所有 False 的位置，并将对应数值强行“抹除”为 0。</p>
      <div id="result-grid-container" class="space-y-3"></div>
    </div>

    <!-- Theory Note -->
    <div class="p-4 bg-amber-950/10 border border-amber-500/20 rounded-2xl">
      <h4 class="text-xs font-bold text-amber-400 mb-2">为什么在 Seq2Seq 中不可或缺？</h4>
      <div class="grid grid-cols-2 gap-4 text-[10px] text-slate-400 leading-relaxed">
        <div class="p-2 bg-black/20 rounded-lg">● <b>防止噪音</b>：Padding 位不包含语义，若参与 Loss 计算会误导模型。</div>
        <div class="p-2 bg-black/20 rounded-lg">● <b>梯度截断</b>：被 Mask 为 0 的位置在反向传播时不会产生梯度。</div>
      </div>
    </div>
  </div>

  <script>
    window.onerror = function(msg) {
        google.colab.kernel.invokeFunction('report_js_error', [msg], {});
    };

    const data = DATA_PLACEHOLDER;

    function render() {
      const maskContainer = document.getElementById('mask-grid-container');
      const resultContainer = document.getElementById('result-grid-container');

      data.valid_lens.forEach((vLen, rowIdx) => {
        // Render Step 1 Matrix
        const mRow = document.createElement('div');
        mRow.className = 'flex items-center gap-6';
        mRow.innerHTML = `<span class="text-[10px] text-slate-600 font-mono w-24">句子 ${rowIdx+1} (len:${vLen})</span>`;

        const mCells = document.createElement('div');
        mCells.className = 'flex gap-2';
        for(let i=0; i<data.maxlen; i++) {
          const isTrue = i < vLen;
          mCells.innerHTML += `<div class="tensor-cell ${isTrue ? 'mask-true' : 'mask-false'}">${isTrue ? 'T' : 'F'}</div>`;
        }
        mRow.appendChild(mCells);
        maskContainer.appendChild(mRow);

        // Render Step 2 Result
        const rRow = document.createElement('div');
        rRow.className = 'flex items-center gap-6';
        rRow.innerHTML = `<span class="text-[10px] text-slate-600 font-mono w-24">处理后 X[${rowIdx}]</span>`;

        const rCells = document.createElement('div');
        rCells.className = 'flex gap-2';
        data.tensor_x[rowIdx].forEach((val, i) => {
          const isMasked = i >= vLen;
          rCells.innerHTML += `<div class="tensor-cell ${isMasked ? 'val-muted' : 'val-active'}">${isMasked ? 0 : val}</div>`;
        });
        rRow.appendChild(rCells);
        resultContainer.appendChild(rRow);
      });
    }

    render();
  </script>
</body>
</html>
""".replace('DATA_PLACEHOLDER', json.dumps(mask_data))

HTML(dashboard_html)

通过扩展softmax交叉熵损失函数来遮蔽不相关的预测

In [ ]:
import torch
from torch import nn

# 1. 扩展 Softmax 交叉熵损失函数类
# 继承自 nn.CrossEntropyLoss，用于在 Seq2Seq 任务中忽略无效的 Padding 词元
class MaskedSoftmaxCELoss(nn.CrossEntropyLoss):
    def forward(self, pred, label, valid_len):
        """pred: 预测值，形状为 (batch_size, seq_len, vocab_size)
        label: 真实标签，形状为 (batch_size, seq_len)
        valid_len: 句子的有效长度，形状为 (batch_size,)"""
        # 第一步：根据有效长度创建一个 0/1 掩码矩阵 (weights)
        # 形状与 label 相同 [B, S]，有效位为 1，Padding 位为 0
        weights = torch.ones_like(label)
        weights = sequence_mask(weights, valid_len)

        # 第二步：关闭内置的 Reduction（求和或求均值）
        # 关键点：我们需要拿到每个词元独立的损失值，以便后续手动加权
        self.reduction = 'none'

        # 第三步：计算原始损失
        # 注意：nn.CrossEntropyLoss 期望的 pred 形状是 (N, C, L)，即 (Batch, Class, Length)
        # 我们的 pred 是 (B, S, V)，所以必须通过 .permute(0, 2, 1) 把 VocabSize 挪到第 2 维
        unweighted_loss = super().forward(pred.permute(0, 2, 1), label)

        # 第四步：加权屏蔽与聚合
        # 将原始损失矩阵与 0/1 权重矩阵点乘，Padding 位置的损失会变成 0
        # 随后在 seq_len (dim=1) 维度求平均，得到每个样本的加权平均损失
        weighted_loss = (unweighted_loss * weights).mean(dim=1)
        return weighted_loss

# --- 模拟测试与原理解析 ---
# 假设：Batch=1, 句子最大长度=3, 词表大小=2
# 设定 valid_len=2，表示前两个词有效，第三个词是 Padding
pred = torch.tensor([[[2.0, 0.5], [0.1, 3.0], [1.0, 1.0]]], requires_grad=True)
label = torch.tensor([[0, 1, 0]]) # 模拟目标标签
valid_len = torch.tensor([2])    # 有效长度只有 2

loss_func = MaskedSoftmaxCELoss()
final_loss = loss_func(pred, label, valid_len)

print(f"[1] 预测矩阵 (3个时间步):\n{pred[0].detach()}")
print(f"[2] 掩码后的句子平均损失: {final_loss.item():.4f}")

# --- 梯度物理效应验证 ---
# 执行反向传播，观察 Padding 位置是否被彻底屏蔽
final_loss.backward()
grads = pred.grad

print("\n--- 梯度屏蔽验证 ---")
print(f"第 1 个词 (有效): {grads[0][0]} -> 有正常梯度")
print(f"第 2 个词 (有效): {grads[0][1]} -> 有正常梯度")
# 这里的梯度必须全为 0，说明 Padding 对模型更新没有任何贡献
print(f"第 3 个词 (Padding): {grads[0][2]} -> 梯度全为 0 (掩码成功)")

[1] 预测矩阵 (3个时间步):
tensor([[2.0000, 0.5000],
        [0.1000, 3.0000],
        [1.0000, 1.0000]])
[2] 掩码后的句子平均损失: 0.0850

--- 梯度屏蔽验证 ---
第 1 个词 (有效): tensor([-0.0608,  0.0608]) -> 有正常梯度
第 2 个词 (有效): tensor([ 0.0174, -0.0174]) -> 有正常梯度
第 3 个词 (Padding): tensor([-0., 0.]) -> 梯度全为 0 (掩码成功)


In [ ]:
# @title
import IPython
from IPython.display import display, HTML

# 纯原生高级交互式：Masked Softmax Cross Entropy Loss 教学面板
html_code = """
<!DOCTYPE html>
<html lang="zh-CN">
<head>
    <meta charset="UTF-8">
    <style>
        :root {
            --bg: #0a0c12; --card: #12151e; --card2: #1a1d2a; --border: #1e2235;
            --purple: #8b7cf8; --orange: #fb923c; --cyan: #22d3ee; --green: #4ade80;
            --red: #f87171; --yellow: #fbbf24; --text: #dde1f0; --muted: #6e7594;
        }
        body { background-color: var(--bg); color: var(--text); font-family: system-ui, -apple-system, sans-serif; margin: 0; padding: 16px; }
        .container { max-width: 820px; margin: 0 auto; display: flex; flex-direction: column; gap: 16px; }
        .header { text-align: center; margin-bottom: 4px; }
        .title-tag { font-size: 10px; color: var(--purple); letter-spacing: 3px; font-weight: 800; text-transform: uppercase; }
        .title { margin: 4px 0; font-size: 20px; font-weight: 800; color: #fff; }
        .subtitle { font-size: 12px; color: var(--muted); margin: 0; }

        /* 导航栏 */
        .tabs { display: grid; grid-template-columns: repeat(4, 1fr); gap: 6px; background: var(--card); padding: 4px; border-radius: 10px; border: 1px solid var(--border); }
        .tab-btn { padding: 10px 4px; border: none; background: transparent; color: var(--muted); font-weight: 700; font-size: 12px; cursor: pointer; border-radius: 8px; transition: all 0.2s; }
        .tab-btn.active { background: var(--purple); color: #fff; box-shadow: 0 2px 6px rgba(139, 124, 248, 0.4); }

        /* 卡片主容器 */
        .card { background: var(--card); border: 1px solid var(--border); border-radius: 14px; padding: 20px; box-shadow: 0 6px 16px rgba(0,0,0,0.4); display: none; flex-direction: column; gap: 14px; }
        .card.active { display: flex; }

        .pill { background: rgba(139, 124, 248, 0.15); color: var(--purple); border: 1px solid rgba(139, 124, 248, 0.35); border-radius: 6px; padding: 2px 8px; font-size: 11px; font-weight: 700; width: fit-content; }
        .code-span { font-family: monospace; color: var(--cyan); background: var(--card2); padding: 2px 6px; border-radius: 4px; font-size: 11px; border: 1px solid var(--border); }
        .code-box { background: var(--bg); padding: 12px; border-radius: 8px; font-family: monospace; font-size: 12px; border-left: 3px solid var(--purple); overflow-x: auto; white-space: pre; }

        /* 交互选择器 */
        .control-panel { background: var(--card2); border: 1px solid var(--border); border-radius: 10px; padding: 12px; display: flex; flex-direction: column; gap: 10px; }
        .slider-row { display: flex; align-items: center; justify-content: space-between; gap: 16px; }
        .slider-label { font-size: 13px; font-weight: bold; width: 220px; }
        .slider-input { flex: 1; accent-color: var(--purple); }
        .slider-val { font-family: monospace; font-size: 14px; color: var(--yellow); width: 40px; text-align: right; }

        /* 矩阵布局 */
        .matrix-block { background: var(--bg); border: 1px solid var(--border); border-radius: 8px; padding: 10px; text-align: center; }
        .matrix-title { font-size: 11px; color: var(--muted); margin-bottom: 6px; font-weight: bold; }
        .grid-matrix { display: grid; grid-template-columns: repeat(4, 1fr); gap: 6px; font-family: monospace; font-size: 12px; }
        .grid-cell { padding: 8px 4px; border-radius: 4px; border: 1px solid var(--border); background: var(--card2); transition: all 0.3s; }

        .math-sign { font-size: 20px; font-weight: bold; color: var(--muted); display: flex; align-items: center; justify-content: center; }
        .matrix-container { display: grid; grid-template-columns: 1fr 40px 1fr 40px 1fr; gap: 8px; align-items: center; }

        /* 最终结果卡片 */
        .res-box { background: var(--card2); border: 1px solid var(--border); border-radius: 8px; padding: 12px; display: flex; justify-content: space-around; font-family: monospace; }
        .res-item { text-align: center; }
        .res-val { font-size: 18px; font-weight: bold; margin-top: 4px; }
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <div class="title-tag">D2L ADVANCED CORE VISUALIZER</div>
            <h1 class="title">MaskedSoftmaxCELoss 动态遮蔽解析器</h1>
            <p class="subtitle">解密 Seq2Seq 批处理中如何通过有效长度（valid_len）瞬间抹除 Padding 噪声</p>
        </div>

        <div class="tabs">
            <button class="tab-btn active" onclick="switchTab('intro')">💡 核心直觉</button>
            <button class="tab-btn" onclick="switchTab('dim')">📐 维度置换之谜</button>
            <button class="tab-btn" onclick="switchTab('sandbox')">🎮 动态计算演练场</button>
            <button class="tab-btn" onclick="switchTab('code')">💻 源码逐行精批</button>
        </div>

        <div id="tab-intro" class="card active">
            <h3 style="margin: 0; font-size: 15px; color: #fff;">🎯 真实业务场景：机器翻译中的填补（Padding）</h3>
            <p style="font-size: 13px; color: var(--muted); margin: 0; line-height: 1.5;">
                假设我们有一个 Batch，包含 2 个句子，最大序列长度设为 4。
                <br>• <b>句子 1</b>: <span class="code-span" style="color:var(--green)">"我" "爱" "你" &lt;pad&gt;</span> ➔ 实际有效长度 = <b>3</b>
                <br>• <b>句子 2</b>: <span class="code-span" style="color:var(--green)">"他" "走" &lt;pad&gt; &lt;pad&gt;</span> ➔ 实际有效长度 = <b>2</b>
            </p>
            <p style="font-size: 13px; color: var(--muted); margin: 4px 0 0 0; line-height: 1.5;">
                对那些明知道是凑数的 <span class="code-span" style="color:var(--red)">&lt;pad&gt;</span>，传统的损失函数依然会疯狂算错并让模型去改权重。而 <b>Mask 掩码矩阵</b> 就像是一个精密的网格漏斗，只放行前 3 项和前 2 项的真实物理损失。
            </p>
            <div style="background: var(--card2); padding: 12px; border-radius: 8px; border-left: 4px solid var(--orange); font-size: 12px;">
                💡 <b>核心计算公式背后的秘密：</b><br>
                <span class="code-span">weighted_loss = unweighted_loss * weights</span> <br>
                当项是有效字符时，<span class="code-span">weight = 1.0</span> ➔ 损失被完整保留；<br>
                当项是填补占位时，<span class="code-span">weight = 0.0</span> ➔ <span style="color:var(--red); font-weight:bold;">Loss 乘零直接人间蒸发！</span>
            </div>
        </div>

        <div id="tab-dim" class="card">
            <h3 style="margin: 0; font-size: 15px;">🧩 为什么要执行 pred.permute(0, 2, 1)？</h3>
            <p style="font-size: 13px; color: var(--muted); margin: 0; line-height: 1.5;">
                这是初学者在阅读李沐老师这段源码时最容易崩溃、卡壳的地方。为什么非要把张量的维度调换位置？
            </p>
            <div class="code-box" style="border-color: var(--cyan);">
# 模型输出的原始 pred 形状：[BatchSize, SeqLen, VocabSize]
# PyTorch 官方 nn.CrossEntropyLoss 对多维输入的硬性要求：
# 官方标准形状必须是： [BatchSize, ClassNDim, OptionalD1, OptionalD2...]
# 对应这里，类别数(VocabSize)必须强行塞在第 1 维！</div>
            <p style="font-size: 13px; color: var(--muted); margin: 4px 0 0 0; line-height: 1.5;">
                因此，必须使用 <span class="code-span">pred.permute(0, 2, 1)</span> 调换第 1 维和第 2 维的物理轴：
                <br>调换前：<span class="code-span">[BatchSize=2, SeqLen=4, VocabSize=5000]</span>
                <br>调换后：<span class="code-span" style="color:var(--green)">[BatchSize=2, VocabSize=5000, SeqLen=4]</span> ➔ 完美适配 PyTorch 底层 C++ 原生算子！
            </p>
        </div>

        <div id="tab-sandbox" class="card">
            <div class="pill">实时运算沙箱引擎（BatchSize=2, MaxSeqLen=4）</div>

            <div class="control-panel">
                <div class="slider-row">
                    <div class="slider-label">句子 1 有效长度 (valid_len[0])：</div>
                    <input type="range" class="slider-input" id="vlen-0" min="1" max="4" value="3" oninput="runMathEngine()">
                    <div class="slider-val" id="vlen-0-txt">3</div>
                </div>
                <div class="slider-row">
                    <div class="slider-label">句子 2 有效长度 (valid_len[1])：</div>
                    <input type="range" class="slider-input" id="vlen-1" min="1" max="4" value="2" oninput="runMathEngine()">
                    <div class="slider-val" id="vlen-1-txt">2</div>
                </div>
            </div>

            <div class="matrix-container">
                <div class="matrix-block">
                    <div class="matrix-title">💥 原始未加权 Loss</div>
                    <div class="grid-matrix" id="mat-raw">
                        </div>
                </div>
                <div class="math-sign">×</div>
                <div class="matrix-block" style="border-color: var(--purple);">
                    <div class="matrix-title" style="color:var(--purple)">🛡️ 动态掩码 Weights</div>
                    <div class="grid-matrix" id="mat-mask">
                        </div>
                </div>
                <div class="math-sign">=</div>
                <div class="matrix-block" style="border-color: var(--green);">
                    <div class="matrix-title" style="color:var(--green)">🎯 掩码后真实 Loss</div>
                    <div class="grid-matrix" id="mat-final">
                        </div>
                </div>
            </div>

            <div style="font-size:12px; color:var(--muted); margin: 4px 0 -4px 0;">
                执行代码末尾的 <span class="code-span">.mean(dim=1)</span>：将序列轴上的有效残余损失在全长(4)上做平滑平均。
            </div>
            <div class="res-box">
                <div class="res-item">
                    <div style="color: var(--cyan); font-size: 11px;">句子 1 最终加权 Loss</div>
                    <div class="res-val" id="res-0" style="color: var(--cyan);">0.00</div>
                </div>
                <div class="res-item">
                    <div style="color: var(--orange); font-size: 11px;">句子 2 最终加权 Loss</div>
                    <div class="res-val" id="res-1" style="color: var(--orange);">0.00</div>
                </div>
            </div>
        </div>

        <div id="tab-code" class="card">
            <h3 style="margin: 0; font-size: 15px;">💻 源码逐行逐句精细级剖析</h3>
            <div class="code-box">
<span style="color:var(--orange)">class</span> <span style="color:var(--green)">MaskedSoftmaxCELoss</span>(nn.CrossEntropyLoss):
    <span style="color:var(--muted)">\"\"\"带遮蔽的 softmax 交叉熵损失函数\"\"\"</span>

    <span style="color:var(--orange)">def</span> <span style="color:var(--purple)">forward</span>(<span style="color:var(--cyan)">self, pred, label, valid_len</span>):
        <span style="color:var(--muted)"># 1. 动态构建一个和标签尺寸一模一样的全 1 矩阵（形状：[BatchSize, SeqLen]）</span>
        weights = torch.ones_like(label)

        <span style="color:var(--muted)"># 2. 调用 D2L 的基础设施内置函数，把超出有效长度范围的格子强行刷写成 0 </span>
        weights = sequence_mask(weights, valid_len)

        <span style="color:var(--muted)"># 3. 关键点！显式宣告不进行任何自动 reduction 聚合（既不求和也不求均值）</span>
        <span style="color:var(--muted)"># 必须保留完整的原子级逐字矩阵： [BatchSize, SeqLen] 以便后面手动点乘 mask</span>
        <span style="color:var(--cyan)">self</span>.reduction = <span style="color:var(--yellow)">'none'</span>

        <span style="color:var(--muted)"># 4. 经典轴置换。把 pred 转换为 [BatchSize, VocabSize, SeqLen] 灌入父类原生前向计算</span>
        unweighted_loss = <span style="color:var(--purple)">super</span>().forward(pred.permute(<span style="color:var(--purple)">0, 2, 1</span>), label)

        <span style="color:var(--muted)"># 5. 点对点点乘！将未加权损失和我们的 0/1 遮蔽矩阵相乘，并沿时间轴 dim=1 计算平均值</span>
        weighted_loss = (unweighted_loss * weights).mean(dim=<span style="color:var(--purple)">1</span>)

        <span style="color:var(--orange)">return</span> weighted_loss</div>
        </div>
    </div>

    <script>
        // 固定基础损失，以防随机抖动导致体验混乱
        const rawLoss = [
            [1.25, 2.40, 0.85, 3.10], // 句子 1 各个时间步对应的原始 CrossEntropyLoss
            [1.95, 0.65, 2.80, 1.45]  // 句子 2 各个时间步对应的原始 CrossEntropyLoss
        ];

        function switchTab(tabId) {
            document.querySelectorAll('.tab-btn').forEach(b => b.classList.remove('active'));
            document.querySelectorAll('.card').forEach(c => c.classList.remove('active'));
            event.target.classList.add('active');
            document.getElementById('tab-' + tabId).classList.add('active');
            if(tabId === 'sandbox') runMathEngine();
        }

        function runMathEngine() {
            // 获取用户拖动的实时有效长度值
            let v0 = parseInt(document.getElementById('vlen-0').value);
            let v1 = parseInt(document.getElementById('vlen-1').value);

            document.getElementById('vlen-0-txt').innerText = v0;
            document.getElementById('vlen-1-txt').innerText = v1;

            let rawHtml = '';
            let maskHtml = '';
            let finalHtml = '';

            let sum0 = 0, sum1 = 0;

            for(let b=0; b<2; b++) {
                let vlen = (b===0) ? v0 : v1;
                for(let s=0; s<4; s++) {
                    let rLoss = rawLoss[b][s];
                    let maskVal = (s < vlen) ? 1.0 : 0.0;
                    let fLoss = rLoss * maskVal;

                    if(b===0) sum0 += fLoss;
                    else sum1 += fLoss;

                    // 样式控制
                    let activeStyle = maskVal === 0.0 ? 'style="opacity: 0.25; border-color: var(--red); color: var(--red); text-decoration: line-through;"' : '';
                    let maskStyle = maskVal === 0.0 ? 'style="color: var(--red); font-weight: bold;"' : 'style="color: var(--green); font-weight: bold;"';
                    let finalStyle = maskVal === 0.0 ? 'style="color: var(--red); font-weight: bold; background: rgba(248,113,113,0.05)"' : 'style="color: var(--green); font-weight: bold; background: rgba(74,222,128,0.05)"';

                    rawHtml += `<div class="grid-cell" ${activeStyle}>${rLoss.toFixed(2)}</div>`;
                    maskHtml += `<div class="grid-cell" ${maskStyle}>${maskVal.toFixed(1)}</div>`;
                    finalHtml += `<div class="grid-cell" ${finalStyle}>${fLoss.toFixed(2)}</div>`;
                }
            }

            // 更新 DOM
            document.getElementById('mat-raw').innerHTML = rawHtml;
            document.getElementById('mat-mask').innerHTML = maskHtml;
            document.getElementById('mat-final').innerHTML = finalHtml;

            // 计算沿 dim=1(全长4) 取 mean 后的值
            document.getElementById('res-0').innerText = (sum0 / 4).toFixed(3);
            document.getElementById('res-1').innerText = (sum1 / 4).toFixed(3);
        }

        // 首次初始化运行
        runMathEngine();
    </script>
</body>
</html>
"""

# 交付 IPython 管道进行完美沙箱隔离渲染
display(HTML(html_code))


代码健全性检查

In [ ]:
loss = MaskedSoftmaxCELoss()
# 修正：去掉不支持的 dtype 参数，并将 valid_len 正确作为第三个参数传入
result = loss(torch.ones(3, 4, 10), torch.ones(3, 4, dtype=torch.long), torch.tensor([4, 2, 0]))
print(result)

tensor([2.3026, 1.1513, 0.0000])


注意：完整训练预测的后续代码这里没有体现 因为没太自己敲 具体看讲义


---

注意力机制